In [1]:
# ============================================================
# CELL 1 - Mount Google Drive
# ============================================================

from google.colab import drive

drive.mount(
    "/content/drive",
    force_remount=False,
)

print("Google Drive mounted successfully.")

Mounted at /content/drive
Google Drive mounted successfully.


In [18]:
# ============================================================
# CELL 2 - Restore SSH key and verify project files
# ============================================================

from pathlib import Path
import os
import shutil

DRIVE_ROOT = Path("/content/drive/MyDrive")

REMOTE_EXEC = DRIVE_ROOT / "remote_exec.sh"
DRIVE_KEY = DRIVE_ROOT / "hadoop-hdfs.pem"
LOCAL_KEY = Path("/content/hadoop-hdfs.pem")

required_files = [
    REMOTE_EXEC,
    DRIVE_KEY,
]

for path in required_files:
    if not path.is_file():
        raise FileNotFoundError(
            f"Required project file not found: {path}"
        )

shutil.copy2(DRIVE_KEY, LOCAL_KEY)

os.chmod(LOCAL_KEY, 0o600)
os.chmod(REMOTE_EXEC, 0o700)

print("Remote execution script:", REMOTE_EXEC)
print("Drive SSH key:", DRIVE_KEY)
print("Local SSH key:", LOCAL_KEY)
print()
print("SSH key restored successfully.")

Remote execution script: /content/drive/MyDrive/remote_exec.sh
Drive SSH key: /content/drive/MyDrive/hadoop-hdfs.pem
Local SSH key: /content/hadoop-hdfs.pem

SSH key restored successfully.


In [19]:
# ============================================================
# CELL 3 - Inspect AWS analytics environment
# ============================================================

import subprocess

REMOTE_EXEC = "/content/drive/MyDrive/remote_exec.sh"

remote_command = r'''
set +e

export HADOOP_HOME=/usr/local/hadoop
export PATH="$HADOOP_HOME/bin:$HADOOP_HOME/sbin:$PATH"

echo "================================================================================"
echo "SYSTEM"
echo "================================================================================"

echo "Hostname:"
hostname

echo
echo "Operating system:"
grep -E '^(PRETTY_NAME|VERSION_ID)=' /etc/os-release

echo
echo "Architecture:"
uname -m

echo
echo "Kernel:"
uname -r

echo
echo "================================================================================"
echo "COMPUTE RESOURCES"
echo "================================================================================"

echo "CPU:"
lscpu | grep -E \
    'Model name|Socket|Core|Thread|CPU\(s\)|Architecture' \
    | head -10

echo
echo "Memory:"
free -h

echo
echo "Disk:"
df -h /

echo
echo "================================================================================"
echo "JAVA"
echo "================================================================================"

if command -v java >/dev/null 2>&1; then
    echo "Java path:"
    command -v java

    echo
    echo "Java version:"
    java -version 2>&1
else
    echo "Java is not installed or not available in PATH."
fi

echo
echo "JAVA_HOME:"
echo "${JAVA_HOME:-NOT_SET}"

echo
echo "================================================================================"
echo "PYTHON"
echo "================================================================================"

if command -v python3 >/dev/null 2>&1; then
    echo "Python path:"
    command -v python3

    echo
    echo "Python version:"
    python3 --version

    echo
    echo "Pip version:"
    python3 -m pip --version 2>&1

    echo
    echo "Installed Python packages of interest:"
    python3 -m pip show \
        pyspark \
        apache-sedona \
        rasterio \
        psutil \
        pandas \
        pyarrow \
        2>/dev/null \
        | grep -E '^(Name|Version|Location):' \
        || true
else
    echo "Python 3 is not installed or not available in PATH."
fi

echo
echo "================================================================================"
echo "SPARK"
echo "================================================================================"

echo "Spark commands:"

for command_name in \
    spark-submit \
    spark-shell \
    pyspark
do
    printf "%-20s" "$command_name"

    if command -v "$command_name" >/dev/null 2>&1; then
        command -v "$command_name"
    else
        echo "NOT_FOUND"
    fi
done

echo
echo "Spark directories:"

for directory in \
    /opt/spark \
    /usr/local/spark \
    /home/ubuntu/spark
do
    if [ -d "$directory" ]; then
        echo "$directory"
    fi
done

echo
echo "Spark version:"

if command -v spark-submit >/dev/null 2>&1; then
    spark-submit --version 2>&1
else
    echo "spark-submit is not currently available in PATH."
fi

echo
echo "PySpark import test:"

python3 - <<'PYTHON_CHECK'
try:
    import pyspark

    print("PySpark import: SUCCESS")
    print("PySpark version:", pyspark.__version__)
    print("PySpark location:", pyspark.__file__)
except Exception as error:
    print("PySpark import: FAILED")
    print(type(error).__name__ + ":", error)
PYTHON_CHECK

echo
echo "================================================================================"
echo "SCALA"
echo "================================================================================"

if command -v scala >/dev/null 2>&1; then
    echo "Scala path:"
    command -v scala

    echo
    echo "Scala version:"
    scala -version 2>&1
else
    echo "Standalone Scala command is not available."
fi

echo
echo "Searching Spark JARs for Scala binary version:"

SPARK_JAR_ROOT=""

for directory in \
    /opt/spark/jars \
    /usr/local/spark/jars \
    /home/ubuntu/spark/jars
do
    if [ -d "$directory" ]; then
        SPARK_JAR_ROOT="$directory"
        break
    fi
done

if [ -z "$SPARK_JAR_ROOT" ]; then
    PYSPARK_DIRECTORY="$(
        python3 - <<'PYTHON_PATH'
try:
    import os
    import pyspark
    print(os.path.dirname(pyspark.__file__))
except Exception:
    pass
PYTHON_PATH
    )"

    if [ -d "$PYSPARK_DIRECTORY/jars" ]; then
        SPARK_JAR_ROOT="$PYSPARK_DIRECTORY/jars"
    fi
fi

if [ -n "$SPARK_JAR_ROOT" ]; then
    echo "Spark JAR directory: $SPARK_JAR_ROOT"

    find "$SPARK_JAR_ROOT" \
        -maxdepth 1 \
        -type f \
        \( \
            -name 'scala-library-*.jar' \
            -o -name 'spark-core_*.jar' \
        \) \
        -printf '%f\n' \
        | sort
else
    echo "Spark JAR directory was not found."
fi

echo
echo "================================================================================"
echo "HADOOP AND HDFS"
echo "================================================================================"

echo "Hadoop version:"
"$HADOOP_HOME/bin/hadoop" version 2>&1 | head -10

echo
echo "Running Hadoop processes:"
jps

echo
echo "HDFS connection:"
"$HADOOP_HOME/bin/hdfs" getconf -confKey fs.defaultFS

echo
echo "HDFS raw-zone summary:"
"$HADOOP_HOME/bin/hdfs" dfs \
    -du -s -h /data/ghsl/raw/india

echo
echo "HDFS GeoTIFF count:"
"$HADOOP_HOME/bin/hdfs" dfs \
    -ls -R /data/ghsl/raw/india \
    | awk '$1 ~ /^-/ && tolower($NF) ~ /\.tiff?$/ {count++} END {print count+0}'

echo
echo "HDFS capacity:"
"$HADOOP_HOME/bin/hdfs" dfs -df -h /

echo
echo "================================================================================"
echo "EXISTING SEDONA INSTALLATION"
echo "================================================================================"

python3 - <<'SEDONA_CHECK'
try:
    import sedona

    print("Sedona Python import: SUCCESS")
    print("Sedona location:", sedona.__file__)

    try:
        from importlib.metadata import version
        print("Sedona version:", version("apache-sedona"))
    except Exception as error:
        print("Could not determine Sedona version:", error)

except Exception as error:
    print("Sedona Python import: NOT INSTALLED")
    print(type(error).__name__ + ":", error)
SEDONA_CHECK

echo
echo "Existing Sedona JARs:"

if [ -n "$SPARK_JAR_ROOT" ]; then
    find "$SPARK_JAR_ROOT" \
        -maxdepth 1 \
        -type f \
        -iname '*sedona*.jar' \
        -printf '%f\n' \
        | sort

    if ! find "$SPARK_JAR_ROOT" \
        -maxdepth 1 \
        -type f \
        -iname '*sedona*.jar' \
        | grep -q .
    then
        echo "No Sedona JARs found."
    fi
else
    echo "Spark JAR directory unavailable."
fi

echo
echo "================================================================================"
echo "DIAGNOSTIC COMPLETE"
echo "================================================================================"
'''

result = subprocess.run(
    ["bash", REMOTE_EXEC, remote_command],
    text=True,
    capture_output=True,
)

print(result.stdout)

if result.stderr:
    print("REMOTE STDERR:")
    print(result.stderr)

print("Return code:", result.returncode)

if result.returncode != 0:
    raise RuntimeError(
        "AWS environment diagnostic failed with return code "
        f"{result.returncode}."
    )

SYSTEM
Hostname:
ip-172-31-30-94

Operating system:
PRETTY_NAME="Ubuntu 26.04 LTS"
VERSION_ID="26.04"

Architecture:
x86_64

Kernel:
7.0.0-1016-aws

COMPUTE RESOURCES
CPU:
Architecture:                            x86_64
CPU(s):                                  2
On-line CPU(s) list:                     0,1
Model name:                              Intel(R) Xeon(R) Platinum 8488C
Thread(s) per core:                      2
Core(s) per socket:                      1
Socket(s):                               1
NUMA node0 CPU(s):                       0,1

Memory:
               total        used        free      shared  buff/cache   available
Mem:           7.6Gi       497Mi       7.1Gi       2.9Mi       297Mi       7.1Gi
Swap:             0B          0B          0B

Disk:
Filesystem      Size  Used Avail Use% Mounted on
/dev/root        77G   16G   62G  21% /

JAVA
Java path:
/usr/bin/java

Java version:
openjdk version "11.0.32.1" 2026-08-18
OpenJDK Runtime Environment (build 11.0.32.1+1-p

In [20]:
# ============================================================
# Start HDFS after EC2 restart and verify GHSL data
# ============================================================

import subprocess

REMOTE_EXEC = "/content/drive/MyDrive/remote_exec.sh"

remote_command = r'''
set -e

export HADOOP_HOME=/usr/local/hadoop
export PATH="$HADOOP_HOME/bin:$HADOOP_HOME/sbin:$PATH"

# Set JAVA_HOME for the non-interactive SSH shell
if [ -z "$JAVA_HOME" ]; then
    JAVA_PATH="$(readlink -f "$(command -v java)")"
    export JAVA_HOME="$(dirname "$(dirname "$JAVA_PATH")")"
fi

echo "HADOOP_HOME: $HADOOP_HOME"
echo "JAVA_HOME:   $JAVA_HOME"

echo
echo "================================================================================"
echo "STARTING HDFS"
echo "================================================================================"

"$HADOOP_HOME/sbin/start-dfs.sh"

sleep 8

echo
echo "================================================================================"
echo "HADOOP PROCESSES"
echo "================================================================================"

jps

echo
echo "================================================================================"
echo "HDFS SAFE MODE"
echo "================================================================================"

"$HADOOP_HOME/bin/hdfs" dfsadmin -safemode get

echo
echo "================================================================================"
echo "GHSL FILE COUNT"
echo "================================================================================"

"$HADOOP_HOME/bin/hdfs" dfs \
    -ls -R /data/ghsl/raw/india \
    | awk '$1 ~ /^-/ && tolower($NF) ~ /\.tiff?$/ {count++} END {print count+0}'

echo
echo "================================================================================"
echo "GHSL DATA SIZE"
echo "================================================================================"

"$HADOOP_HOME/bin/hdfs" dfs \
    -du -s -h /data/ghsl/raw/india

echo
echo "================================================================================"
echo "HDFS CAPACITY"
echo "================================================================================"

"$HADOOP_HOME/bin/hdfs" dfs -df -h /

echo
echo "================================================================================"
echo "HDFS HEALTH"
echo "================================================================================"

"$HADOOP_HOME/bin/hdfs" fsck /data/ghsl/raw/india \
    | grep -E \
    "Status:|Total size:|Total files:|Total blocks|Missing blocks:|Missing replicas:"
'''

result = subprocess.run(
    ["bash", REMOTE_EXEC, remote_command],
    text=True,
    capture_output=True,
)

print(result.stdout)

if result.stderr:
    print("REMOTE STDERR:")
    print(result.stderr)

if result.returncode != 0:
    raise RuntimeError(
        "HDFS startup or verification failed with return code "
        f"{result.returncode}."
    )

HADOOP_HOME: /usr/local/hadoop
JAVA_HOME:   /usr/lib/jvm/java-11-openjdk-amd64

STARTING HDFS
Starting namenodes on [localhost]
Starting datanodes
Starting secondary namenodes [ip-172-31-30-94]

HADOOP PROCESSES
1813 DataNode
2039 SecondaryNameNode
1673 NameNode
2186 Jps

HDFS SAFE MODE
Safe mode is ON

GHSL FILE COUNT
325

GHSL DATA SIZE
6.6 G  6.6 G  /data/ghsl/raw/india

HDFS CAPACITY
Filesystem               Size   Used  Available  Use%
hdfs://localhost:9000  76.4 G  6.6 G     61.1 G    9%

HDFS HEALTH
Status: HEALTHY
 Total size:	7038596744 B
 Total files:	325
 Total blocks (validated):	325 (avg. block size 21657220 B)
 Missing blocks:		0
 Missing replicas:		0 (0.0 %)
 Total size:	0 B
 Total files:	0

REMOTE STDERR:
Connecting to namenode via http://localhost:9870/fsck?ugi=ubuntu&path=%2Fdata%2Fghsl%2Fraw%2Findia



In [21]:
# ============================================================
# Leave HDFS safe mode and verify write readiness
# ============================================================

import subprocess

REMOTE_EXEC = "/content/drive/MyDrive/remote_exec.sh"

remote_command = r'''
set -e

export HADOOP_HOME=/usr/local/hadoop
export PATH="$HADOOP_HOME/bin:$HADOOP_HOME/sbin:$PATH"

echo "================================================================================"
echo "CURRENT SAFE MODE STATUS"
echo "================================================================================"

hdfs dfsadmin -safemode get

echo
echo "================================================================================"
echo "LEAVING SAFE MODE"
echo "================================================================================"

hdfs dfsadmin -safemode leave

echo
echo "================================================================================"
echo "FINAL SAFE MODE STATUS"
echo "================================================================================"

hdfs dfsadmin -safemode get

echo
echo "================================================================================"
echo "HDFS WRITE TEST"
echo "================================================================================"

TEST_PATH="/tmp/ghsl_sedona_write_test_$(date +%s).txt"

echo "HDFS write test successful" \
    | hdfs dfs -put -f - "$TEST_PATH"

hdfs dfs -cat "$TEST_PATH"
hdfs dfs -rm "$TEST_PATH"

echo
echo "================================================================================"
echo "FINAL STATUS"
echo "================================================================================"
echo "HDFS is healthy, writable, and ready for Spark/Sedona."
'''

result = subprocess.run(
    ["bash", REMOTE_EXEC, remote_command],
    text=True,
    capture_output=True,
)

print(result.stdout)

if result.stderr:
    print("REMOTE STDERR:")
    print(result.stderr)

if result.returncode != 0:
    raise RuntimeError(
        "HDFS safe-mode or write-readiness check failed with "
        f"return code {result.returncode}."
    )

CURRENT SAFE MODE STATUS
Safe mode is ON

LEAVING SAFE MODE
Safe mode is OFF

FINAL SAFE MODE STATUS
Safe mode is OFF

HDFS WRITE TEST
HDFS write test successful
Deleted /tmp/ghsl_sedona_write_test_1793720452.txt

FINAL STATUS
HDFS is healthy, writable, and ready for Spark/Sedona.

REMOTE STDERR:



In [22]:
# ============================================================
# Install isolated Python 3.12, PySpark, and Apache Sedona
# ============================================================

import subprocess

REMOTE_EXEC = "/content/drive/MyDrive/remote_exec.sh"

remote_command = r"""
set -euo pipefail

ENV_DIR="/home/ubuntu/ghsl-sedona-env"
UV_BIN="/home/ubuntu/.local/bin/uv"

echo "================================================================================"
echo "SYSTEM PREPARATION"
echo "================================================================================"

sudo apt-get update -qq
sudo apt-get install -y -qq curl ca-certificates

echo
echo "================================================================================"
echo "INSTALLING UV"
echo "================================================================================"

if [ ! -x "$UV_BIN" ]; then
    curl -LsSf https://astral.sh/uv/install.sh | sh
fi

"$UV_BIN" --version

echo
echo "================================================================================"
echo "INSTALLING ISOLATED PYTHON 3.12"
echo "================================================================================"

"$UV_BIN" python install 3.12

echo
echo "================================================================================"
echo "CREATING PROJECT ENVIRONMENT"
echo "================================================================================"

if [ ! -x "$ENV_DIR/bin/python" ]; then
    "$UV_BIN" venv \
        --python 3.12 \
        "$ENV_DIR"
fi

PYTHON_BIN="$ENV_DIR/bin/python"

echo "Environment: $ENV_DIR"
echo "Python:      $PYTHON_BIN"

"$PYTHON_BIN" --version

echo
echo "================================================================================"
echo "INSTALLING PROJECT PACKAGES"
echo "================================================================================"

"$UV_BIN" pip install \
    --python "$PYTHON_BIN" \
    "pyspark==3.5.9" \
    "apache-sedona==1.9.1" \
    "rasterio" \
    "numpy" \
    "pandas" \
    "pyarrow" \
    "psutil"

echo
echo "================================================================================"
echo "CREATING ENVIRONMENT CONFIGURATION"
echo "================================================================================"

cat > /home/ubuntu/ghsl-sedona-env.sh <<'ENV_CONFIG'
export JAVA_HOME=/usr/lib/jvm/java-11-openjdk-amd64
export HADOOP_HOME=/usr/local/hadoop
export HADOOP_CONF_DIR=/usr/local/hadoop/etc/hadoop
export PATH=/home/ubuntu/ghsl-sedona-env/bin:/usr/local/hadoop/bin:/usr/local/hadoop/sbin:$PATH
export PYSPARK_PYTHON=/home/ubuntu/ghsl-sedona-env/bin/python
export PYSPARK_DRIVER_PYTHON=/home/ubuntu/ghsl-sedona-env/bin/python
ENV_CONFIG

chmod 700 /home/ubuntu/ghsl-sedona-env.sh
source /home/ubuntu/ghsl-sedona-env.sh

echo "JAVA_HOME:             $JAVA_HOME"
echo "HADOOP_HOME:           $HADOOP_HOME"
echo "HADOOP_CONF_DIR:       $HADOOP_CONF_DIR"
echo "PYSPARK_PYTHON:        $PYSPARK_PYTHON"
echo "PYSPARK_DRIVER_PYTHON: $PYSPARK_DRIVER_PYTHON"

echo
echo "================================================================================"
echo "PACKAGE VERIFICATION"
echo "================================================================================"

"$PYTHON_BIN" - <<'PYTHON_CHECK'
from importlib.metadata import version
import sys

import numpy
import pandas
import psutil
import pyarrow
import pyspark
import rasterio
import sedona

print("Python:", sys.version.split()[0])
print("PySpark:", pyspark.__version__)
print("Apache Sedona:", version("apache-sedona"))
print("Rasterio:", rasterio.__version__)
print("NumPy:", numpy.__version__)
print("Pandas:", pandas.__version__)
print("PyArrow:", pyarrow.__version__)
print("psutil:", psutil.__version__)
print("Sedona module:", sedona.__file__)
PYTHON_CHECK

echo
echo "================================================================================"
echo "SPARK COMMAND VERIFICATION"
echo "================================================================================"

command -v spark-submit
command -v pyspark

spark-submit --version 2>&1 | head -20

echo
echo "================================================================================"
echo "INSTALLATION COMPLETE"
echo "================================================================================"

echo "Python 3.12, PySpark, and Apache Sedona installed successfully."
"""

result = subprocess.run(
    ["bash", REMOTE_EXEC, remote_command],
    text=True,
    capture_output=True,
)

print(result.stdout)

if result.stderr:
    print("REMOTE STDERR:")
    print(result.stderr)

print("Return code:", result.returncode)

if result.returncode != 0:
    raise RuntimeError(
        "Spark/Sedona environment installation failed with "
        f"return code {result.returncode}."
    )

SYSTEM PREPARATION

INSTALLING UV
uv 0.13.6 (x86_64-unknown-linux-gnu)

INSTALLING ISOLATED PYTHON 3.12

CREATING PROJECT ENVIRONMENT
Environment: /home/ubuntu/ghsl-sedona-env
Python:      /home/ubuntu/ghsl-sedona-env/bin/python
Python 3.12.16

INSTALLING PROJECT PACKAGES

CREATING ENVIRONMENT CONFIGURATION
JAVA_HOME:             /usr/lib/jvm/java-11-openjdk-amd64
HADOOP_HOME:           /usr/local/hadoop
HADOOP_CONF_DIR:       /usr/local/hadoop/etc/hadoop
PYSPARK_PYTHON:        /home/ubuntu/ghsl-sedona-env/bin/python
PYSPARK_DRIVER_PYTHON: /home/ubuntu/ghsl-sedona-env/bin/python

PACKAGE VERIFICATION
Python: 3.12.16
PySpark: 3.5.9
Apache Sedona: 1.9.1
Rasterio: 1.5.2
NumPy: 2.5.4
Pandas: 3.1.0
PyArrow: 26.0.0
psutil: 7.2.2
Sedona module: /home/ubuntu/ghsl-sedona-env/lib/python3.12/site-packages/sedona/__init__.py

SPARK COMMAND VERIFICATION
/home/ubuntu/ghsl-sedona-env/bin/spark-submit
/home/ubuntu/ghsl-sedona-env/bin/pyspark
Welcome to
      ____              __
     / __/__  ___ ____

In [23]:
# ============================================================
# Verify HDFS after Spark and Sedona installation
# ============================================================

import subprocess

REMOTE_EXEC = "/content/drive/MyDrive/remote_exec.sh"

remote_command = r"""
set -e

export JAVA_HOME=/usr/lib/jvm/java-11-openjdk-amd64
export HADOOP_HOME=/usr/local/hadoop
export PATH="$HADOOP_HOME/bin:$HADOOP_HOME/sbin:$PATH"

echo "================================================================================"
echo "HADOOP PROCESSES"
echo "================================================================================"

jps

echo
echo "================================================================================"
echo "HDFS SAFE MODE"
echo "================================================================================"

hdfs dfsadmin -safemode get

echo
echo "================================================================================"
echo "GHSL GEOTIFF COUNT"
echo "================================================================================"

hdfs dfs -ls -R /data/ghsl/raw/india \
    | awk '$1 ~ /^-/ && tolower($NF) ~ /\.tiff?$/ {count++} END {print count+0}'

echo
echo "================================================================================"
echo "GHSL DATA SIZE"
echo "================================================================================"

hdfs dfs -du -s -h /data/ghsl/raw/india

echo
echo "================================================================================"
echo "HDFS HEALTH"
echo "================================================================================"

hdfs fsck /data/ghsl/raw/india \
    | grep -E \
    "Status:|Total size:|Total files:|Total blocks|Missing blocks:|Missing replicas:"

echo
echo "================================================================================"
echo "HDFS CAPACITY"
echo "================================================================================"

hdfs dfs -df -h /

echo
echo "================================================================================"
echo "SYSTEM PYTHON REMAINS UNCHANGED"
echo "================================================================================"

echo -n "System Python:  "
/usr/bin/python3 --version

echo -n "Project Python: "
/home/ubuntu/ghsl-sedona-env/bin/python --version

echo
echo "================================================================================"
echo "FINAL STATUS"
echo "================================================================================"

echo "Spark and Sedona installation did not affect HDFS."
"""

result = subprocess.run(
    ["bash", REMOTE_EXEC, remote_command],
    text=True,
    capture_output=True,
)

print(result.stdout)

if result.stderr:
    print("REMOTE STDERR:")
    print(result.stderr)

print("Return code:", result.returncode)

if result.returncode != 0:
    raise RuntimeError(
        "Post-installation HDFS verification failed with "
        f"return code {result.returncode}."
    )

HADOOP PROCESSES
1813 DataNode
2039 SecondaryNameNode
3032 Jps
1673 NameNode

HDFS SAFE MODE
Safe mode is OFF

GHSL GEOTIFF COUNT
325

GHSL DATA SIZE
6.6 G  6.6 G  /data/ghsl/raw/india

HDFS HEALTH
Status: HEALTHY
 Total size:	7038596744 B
 Total files:	325
 Total blocks (validated):	325 (avg. block size 21657220 B)
 Missing blocks:		0
 Missing replicas:		0 (0.0 %)
 Total size:	0 B
 Total files:	0

HDFS CAPACITY
Filesystem               Size   Used  Available  Use%
hdfs://localhost:9000  76.4 G  6.6 G     61.1 G    9%

SYSTEM PYTHON REMAINS UNCHANGED
System Python:  Python 3.14.4
Project Python: Python 3.12.16

FINAL STATUS
Spark and Sedona installation did not affect HDFS.

REMOTE STDERR:
Connecting to namenode via http://localhost:9870/fsck?ugi=ubuntu&path=%2Fdata%2Fghsl%2Fraw%2Findia

Return code: 0


In [24]:
# ============================================================
# Sedona smoke test - safe version without nested shell strings
# ============================================================

from pathlib import Path
import subprocess
import textwrap

REMOTE_EXEC = "/content/drive/MyDrive/remote_exec.sh"
SSH_KEY = "/content/hadoop-hdfs.pem"
AWS_HOST = "54.92.192.154"
AWS_USER = "ubuntu"

LOCAL_SCRIPT = Path("/content/sedona_smoke_test.py")
REMOTE_SCRIPT = "/home/ubuntu/sedona_smoke_test.py"

# ------------------------------------------------------------
# 1. Create the Python test script locally in Colab
# ------------------------------------------------------------

script_content = textwrap.dedent('''
from sedona.spark import SedonaContext

spark = None

try:
    print("=" * 80)
    print("CREATING SPARK AND SEDONA CONTEXT")
    print("=" * 80)

    config = (
        SedonaContext.builder()
        .master("local[2]")
        .appName("GHSL-Sedona-Smoke-Test")
        .config("spark.driver.memory", "2g")
        .config("spark.sql.shuffle.partitions", "4")
        .config("spark.default.parallelism", "2")
        .config("spark.ui.enabled", "false")
        .getOrCreate()
    )

    spark = SedonaContext.create(config)
    spark.sparkContext.setLogLevel("WARN")

    print("Spark version:", spark.version)
    print("Spark master:", spark.sparkContext.master)
    print("Application:", spark.sparkContext.appName)

    print()
    print("=" * 80)
    print("SEDONA SPATIAL SQL TEST")
    print("=" * 80)

    query = (
        "SELECT "
        "ST_AsText(ST_Point(77.5946, 12.9716)) AS point_wkt, "
        "ST_Contains("
        "ST_PolygonFromEnvelope(77.50, 12.90, 77.70, 13.10), "
        "ST_Point(77.5946, 12.9716)"
        ") AS point_inside"
    )

    result = spark.sql(query)
    result.show(truncate=False)

    row = result.first()

    if row["point_inside"] is not True:
        raise RuntimeError("Sedona spatial containment test failed.")

    print("Sedona spatial SQL: SUCCESS")

    print()
    print("=" * 80)
    print("HDFS CONNECTIVITY TEST")
    print("=" * 80)

    configuration = spark.sparkContext._jsc.hadoopConfiguration()
    default_fs = configuration.get("fs.defaultFS")

    jvm = spark.sparkContext._jvm
    hdfs_path = jvm.org.apache.hadoop.fs.Path(
        "/data/ghsl/raw/india"
    )
    filesystem = hdfs_path.getFileSystem(configuration)

    if not filesystem.exists(hdfs_path):
        raise RuntimeError("GHSL HDFS path is not accessible.")

    summary = filesystem.getContentSummary(hdfs_path)

    file_count = summary.getFileCount()
    logical_bytes = summary.getLength()

    print("Spark fs.defaultFS:", default_fs)
    print("HDFS path exists:", True)
    print("HDFS file count:", file_count)
    print("HDFS logical bytes:", logical_bytes)

    if file_count != 325:
        raise RuntimeError(
            f"Expected 325 HDFS files, found {file_count}."
        )

    if logical_bytes != 7038596744:
        raise RuntimeError(
            f"Unexpected HDFS byte count: {logical_bytes}."
        )

    print("Spark-to-HDFS connectivity: SUCCESS")

    print()
    print("=" * 80)
    print("FINAL RESULT")
    print("=" * 80)
    print("SEDONA SMOKE TEST: SUCCESS")

finally:
    if spark is not None:
        spark.stop()
''').strip() + "\n"

LOCAL_SCRIPT.write_text(
    script_content,
    encoding="utf-8",
)

print("Local test script created:", LOCAL_SCRIPT)
print("Script size:", LOCAL_SCRIPT.stat().st_size, "bytes")

# ------------------------------------------------------------
# 2. Copy the test script from Colab to AWS
# ------------------------------------------------------------

scp_result = subprocess.run(
    [
        "scp",
        "-o", "StrictHostKeyChecking=no",
        "-o", "UserKnownHostsFile=/dev/null",
        "-o", "BatchMode=yes",
        "-o", "ConnectTimeout=30",
        "-i", SSH_KEY,
        str(LOCAL_SCRIPT),
        f"{AWS_USER}@{AWS_HOST}:{REMOTE_SCRIPT}",
    ],
    text=True,
    capture_output=True,
)

if scp_result.returncode != 0:
    print(scp_result.stderr)
    raise RuntimeError("Failed to copy the Sedona test script to AWS.")

print("Test script copied to AWS successfully.")

# ------------------------------------------------------------
# 3. Run Spark with matching Sedona dependencies
# ------------------------------------------------------------

remote_command = (
    "source /home/ubuntu/ghsl-sedona-env.sh && "
    "spark-submit "
    "--master local[2] "
    "--packages "
    "org.apache.sedona:sedona-spark-3.5_2.12:1.9.1,"
    "org.datasyslab:geotools-wrapper:1.9.1-33.5 "
    "--repositories "
    "https://artifacts.unidata.ucar.edu/repository/unidata-all "
    "/home/ubuntu/sedona_smoke_test.py"
)

result = subprocess.run(
    ["bash", REMOTE_EXEC, remote_command],
    text=True,
    capture_output=True,
    timeout=900,
)

print(result.stdout)

if result.stderr:
    print("REMOTE STDERR:")
    print(result.stderr)

print("Return code:", result.returncode)

if result.returncode != 0:
    raise RuntimeError(
        f"Sedona smoke test failed with return code "
        f"{result.returncode}."
    )

Local test script created: /content/sedona_smoke_test.py
Script size: 2592 bytes
Test script copied to AWS successfully.
:: loading settings :: url = jar:file:/home/ubuntu/ghsl-sedona-env/lib/python3.12/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
CREATING SPARK AND SEDONA CONTEXT
Spark version: 3.5.9
Spark master: local[2]
Application: GHSL-Sedona-Smoke-Test

SEDONA SPATIAL SQL TEST
+-----------------------+------------+
|point_wkt              |point_inside|
+-----------------------+------------+
|POINT (77.5946 12.9716)|true        |
+-----------------------+------------+

Sedona spatial SQL: SUCCESS

HDFS CONNECTIVITY TEST
Spark fs.defaultFS: hdfs://localhost:9000
HDFS path exists: True
HDFS file count: 325
HDFS logical bytes: 7038596744
Spark-to-HDFS connectivity: SUCCESS

FINAL RESULT
SEDONA SMOKE TEST: SUCCESS

REMOTE STDERR:
https://artifacts.unidata.ucar.edu/repository/unidata-all added as a remote repository with the name: repo-1
Ivy 

Confirmed results
output shows Spark 3.5.9 started successfully in local[2] mode.
Apache Sedona 1.9.1 initialized with the matching Spark 3.5 / Scala 2.12 dependencies.
Sedona spatial SQL returned:
POINT (77.5946 12.9716)
containment result true
Spark connected to hdfs://localhost:9000.
Spark found the GHSL raw zone.
Spark saw exactly 325 files.
Spark saw exactly 7,038,596,744 logical bytes.
The final return code was 0.
Plain Text
SEDONA SPATIAL SQL: SUCCESS
SPARK -> HDFS: SUCCESS
GHSL FILE COUNT: 325
GHSL LOGICAL BYTES: 7,038,596,744
SEDONA SMOKE TEST: SUCCESS
RETURN CODE: 0

The JARs are now cached under:

Plain Text
/home/ubuntu/.ivy2/

So later runs should not repeat all those downloads.

The Unable to load native-hadoop library warning is not a failure. Spark used its built-in Java Hadoop classes, and the HDFS checks succeeded. The repeated Added JAR and Fetching lines are normal Spark dependency setup messages.

Project checkpoint
Plain Text
Apache Sedona installed DONE
Matching Spark/Scala dependencies DONE
Sedona spatial SQL test DONE
Spark-to-HDFS connection DONE
325-file GHSL inventory verified DONE

Next: Read one real GHSL GeoTIFF through Sedona's raster engine

The next test should use one actual GHSL raster from HDFS and extract its:

Plain Text
CRS
width and height
band count
pixel resolution
spatial envelope
NoData value
basic raster statistics

That will move us from “Sedona is configured” to “Sedona is genuinely integrated with the project data.”


In [ ]:
# ============================================================
# Read one real GHSL GeoTIFF using Apache Sedona
# ============================================================

from pathlib import Path
import subprocess

REMOTE_EXEC = "/content/drive/MyDrive/remote_exec.sh"
SSH_KEY = "/content/hadoop-hdfs.pem"

AWS_HOST = "54.92.192.154"
AWS_USER = "ubuntu"

LOCAL_SCRIPT = Path("/content/ghsl_sedona_raster_metadata_test.py")
REMOTE_SCRIPT = "/home/ubuntu/ghsl_sedona_raster_metadata_test.py"

# ------------------------------------------------------------
# 1. Select one real SMOD GeoTIFF from HDFS
# ------------------------------------------------------------

find_command = (
    "export HADOOP_HOME=/usr/local/hadoop; "
    "export PATH=$HADOOP_HOME/bin:$HADOOP_HOME/sbin:$PATH; "
    "hdfs dfs -ls -R /data/ghsl/raw/india/smod "
    "| awk '$1 ~ /^-/ && tolower($NF) ~ /\\.tiff?$/ {print $NF}' "
    "| head -1"
)

find_result = subprocess.run(
    ["bash", REMOTE_EXEC, find_command],
    text=True,
    capture_output=True,
)

if find_result.returncode != 0:
    print(find_result.stderr)
    raise RuntimeError("Could not search the HDFS SMOD directory.")

HDFS_FILE = find_result.stdout.strip()

if not HDFS_FILE:
    raise RuntimeError("No SMOD GeoTIFF was found in HDFS.")

HDFS_URI = f"hdfs://localhost:9000{HDFS_FILE}"

print("Selected HDFS file:")
print(HDFS_FILE)
print()
print("Sedona input URI:")
print(HDFS_URI)

# ------------------------------------------------------------
# 2. Create the Sedona raster metadata test script
# ------------------------------------------------------------

script_content = f'''
from sedona.spark import SedonaContext

HDFS_URI = {HDFS_URI!r}

spark = None

try:
    print("=" * 90)
    print("STARTING APACHE SEDONA")
    print("=" * 90)

    config = (
        SedonaContext.builder()
        .master("local[2]")
        .appName("GHSL-Sedona-Raster-Metadata-Test")
        .config("spark.driver.memory", "3g")
        .config("spark.sql.shuffle.partitions", "4")
        .config("spark.default.parallelism", "2")
        .config("spark.ui.enabled", "false")
        .getOrCreate()
    )

    spark = SedonaContext.create(config)
    spark.sparkContext.setLogLevel("WARN")

    print("Spark version:", spark.version)
    print("Spark master:", spark.sparkContext.master)
    print("GeoTIFF URI:", HDFS_URI)

    print()
    print("=" * 90)
    print("READING REAL GHSL GEOTIFF METADATA")
    print("=" * 90)

    metadata = (
        spark.read
        .format("geotiff.metadata")
        .load(HDFS_URI)
    )

    print("Rows returned:", metadata.count())

    print()
    print("=" * 90)
    print("SEDONA METADATA SCHEMA")
    print("=" * 90)

    metadata.printSchema()

    print()
    print("=" * 90)
    print("CORE RASTER METADATA")
    print("=" * 90)

    selected = metadata.select(
        "path",
        "driver",
        "fileSize",
        "width",
        "height",
        "numBands",
        "srid",
        "crs",
        "geoTransform",
        "cornerCoordinates",
        "bands",
        "overviews",
        "isTiled",
        "compression",
    )

    selected.show(
        n=1,
        truncate=False,
        vertical=True,
    )

    row = metadata.first()

    print()
    print("=" * 90)
    print("VALIDATION")
    print("=" * 90)

    checks = {{
        "One metadata row returned": metadata.count() == 1,
        "Driver is GTiff": row["driver"] == "GTiff",
        "File size is positive": row["fileSize"] > 0,
        "Width is positive": row["width"] > 0,
        "Height is positive": row["height"] > 0,
        "At least one band exists": row["numBands"] >= 1,
        "CRS is present": bool(row["crs"]),
        "GeoTransform is present": row["geoTransform"] is not None,
        "Corner coordinates are present":
            row["cornerCoordinates"] is not None,
    }}

    for description, passed in checks.items():
        print(
            f"{{description:<40}} "
            f"{{'PASS' if passed else 'FAIL'}}"
        )

    failed_checks = [
        description
        for description, passed in checks.items()
        if not passed
    ]

    if failed_checks:
        raise RuntimeError(
            "Metadata validation failed: "
            + ", ".join(failed_checks)
        )

    print()
    print("=" * 90)
    print("FINAL RESULT")
    print("=" * 90)

    print("Real GHSL GeoTIFF read from HDFS using Apache Sedona.")
    print("Raster metadata extracted and validated successfully.")
    print("SEDONA GHSL RASTER METADATA TEST: SUCCESS")

finally:
    if spark is not None:
        spark.stop()
'''.strip() + "\n"

LOCAL_SCRIPT.write_text(
    script_content,
    encoding="utf-8",
)

print()
print("Local test script created:", LOCAL_SCRIPT)
print("Script size:", LOCAL_SCRIPT.stat().st_size, "bytes")

# ------------------------------------------------------------
# 3. Copy the test script to AWS
# ------------------------------------------------------------

scp_result = subprocess.run(
    [
        "scp",
        "-o", "StrictHostKeyChecking=no",
        "-o", "UserKnownHostsFile=/dev/null",
        "-o", "BatchMode=yes",
        "-o", "ConnectTimeout=30",
        "-i", SSH_KEY,
        str(LOCAL_SCRIPT),
        f"{AWS_USER}@{AWS_HOST}:{REMOTE_SCRIPT}",
    ],
    text=True,
    capture_output=True,
)

if scp_result.returncode != 0:
    print(scp_result.stderr)
    raise RuntimeError(
        "Failed to copy the raster metadata test script to AWS."
    )

print("Test script copied to AWS successfully.")

# ------------------------------------------------------------
# 4. Run the real Sedona raster metadata test
# ------------------------------------------------------------

run_command = (
    "source /home/ubuntu/ghsl-sedona-env.sh && "
    "spark-submit "
    "--master local[2] "
    "--packages "
    "org.apache.sedona:sedona-spark-3.5_2.12:1.9.1,"
    "org.datasyslab:geotools-wrapper:1.9.1-33.5 "
    "--repositories "
    "https://artifacts.unidata.ucar.edu/repository/unidata-all "
    f"{REMOTE_SCRIPT}"
)

result = subprocess.run(
    ["bash", REMOTE_EXEC, run_command],
    text=True,
    capture_output=True,
    timeout=900,
)

print()
print(result.stdout)

if result.stderr:
    print("REMOTE STDERR:")
    print(result.stderr)

print("Return code:", result.returncode)

if result.returncode != 0:
    raise RuntimeError(
        "Sedona GHSL raster metadata test failed with "
        f"return code {result.returncode}."
    )

The real GHSL raster metadata test passed completely. output confirms that Apache Sedona successfully read an actual GeoTIFF directly from HDFS, not from Google Drive or the AWS local filesystem.

Confirmed raster integration

The tested file was:

Plain Text
/data/ghsl/raw/india/smod/epoch=1975/
ghsl_india_smod_1975_1000m_r2023a.tif

Sedona read it from:

Plain Text
hdfs://localhost:9000/data/ghsl/raw/india/smod/epoch=1975/
ghsl_india_smod_1975_1000m_r2023a.tif

The test returned exactly one metadata row, all validation checks passed, and the process ended with return code 0.

Extracted metadata
Plain Text
Driver: GTiff
File size: 1,071,124 bytes
Dimensions: 2902 x 3179 pixels
Bands: 1
Data type: Signed 16-bit integer
Color interpretation: Gray
Resolution: 1000 m x 1000 m
Projection: World Mollweide
Tiled internally: Yes
Tile/block size: 256 x 256
Compression: LZW
Overview levels: 5
NoData value: Not explicitly recorded

Sedona extracted the affine transform:

Plain Text
Upper-left X: 6,471,000
Upper-left Y: 4,013,000
Scale X: 1,000
Scale Y: -1,000
Skew X: 0
Skew Y: 0

The negative Y scale is correct for a north-up raster because raster rows progress downward while projected northing decreases. The extracted bounding coordinates were:

Plain Text
minX: 6,471,000
minY: 834,000
maxX: 9,373,000
maxY: 4,013,000

These values are internally consistent:

Plain Text
Width = (9,373,000 - 6,471,000) / 1,000 = 2,902 pixels
Height = (4,013,000 - 834,000) / 1,000 = 3,179 pixels

All of these values came directly from the Sedona result.

About srid = 0

The output shows:

Plain Text
srid: 0
crs: World_Mollweide WKT


This is not a missing CRS. The full Mollweide projection WKT was successfully extracted. srid = 0 means the reader did not map that WKT to a standard EPSG integer identifier. Our GHSL validation had identified the projection as ESRI:54009, but Sedona preserved it as WKT rather than assigning an EPSG-style numeric SRID.

For our project, we should therefore use:

Plain Text
Canonical project CRS: ESRI:54009
Sedona metadata SRID: 0
Sedona CRS WKT: World Mollweide

We must not interpret 0 as an unreferenced raster.

About the overview levels

The file contains five lower-resolution internal overviews:

Plain Text
1451 x 1590
726 x 795
363 x 398
182 x 199
91 x 100

That, together with internal tiling and LZW compression, means the file is organized efficiently for reduced-resolution access. However, we should avoid formally labeling it a Cloud Optimized GeoTIFF based only on this output because complete COG compliance also depends on internal byte ordering and layout requirements.

Current checkpoint
Plain Text
Sedona installation COMPLETE
Sedona spatial SQL COMPLETE
Spark-to-HDFS connectivity COMPLETE
Real GHSL GeoTIFF read from HDFS COMPLETE
CRS and geotransform extraction COMPLETE
Band and raster-layout inspection COMPLETE
Metadata validation COMPLETE

Apache Sedona is now genuinely integrated into the project. The professor’s requested geospatial Spark-engine requirement is no longer only planned or installed. It has been demonstrated against the actual GHSL dataset.

Next task

We should now build the complete Sedona metadata catalog for all 325 GeoTIFF files and save it as a curated Parquet dataset:

Plain Text
/data/ghsl/metadata/india/geotiff_catalog

That catalog will provide one row per physical shard with:

Plain Text
product
epoch/year
path
file size
width
height
band count
CRS
resolution
bounds
data type
NoData
tiling
compression
overview count

It will also give us our first quantitative Sedona measurement:

Files processed
Input bytes scanned
Wall-clock runtime
Files per second
MiB per second
Peak memory
CPU utilization

In [ ]:
# ============================================================
# Build complete Sedona metadata catalog for all 325 GHSL files
# ============================================================

from pathlib import Path
import subprocess
import textwrap

REMOTE_EXEC = "/content/drive/MyDrive/remote_exec.sh"
SSH_KEY = "/content/hadoop-hdfs.pem"

AWS_HOST = "54.92.192.154"
AWS_USER = "ubuntu"

LOCAL_SCRIPT = Path("/content/build_ghsl_sedona_catalog.py")
REMOTE_SCRIPT = "/home/ubuntu/build_ghsl_sedona_catalog.py"

RAW_ROOT = "hdfs://localhost:9000/data/ghsl/raw/india"
CATALOG_PATH = "hdfs://localhost:9000/data/ghsl/metadata/india/geotiff_catalog"
BENCHMARK_PATH = "hdfs://localhost:9000/data/ghsl/metadata/india/catalog_benchmark"

EXPECTED_FILES = 325
EXPECTED_BYTES = 7_038_596_744

# ------------------------------------------------------------
# 1. Create the Spark/Sedona job locally in Colab
# ------------------------------------------------------------

script_content = textwrap.dedent(f'''
import json
import platform
import statistics
import threading
import time
from datetime import datetime, timezone

import psutil

from pyspark.sql import functions as F
from sedona.spark import SedonaContext


RAW_ROOT = {RAW_ROOT!r}
CATALOG_PATH = {CATALOG_PATH!r}
BENCHMARK_PATH = {BENCHMARK_PATH!r}

EXPECTED_FILES = {EXPECTED_FILES}
EXPECTED_BYTES = {EXPECTED_BYTES}

spark = None
monitor_stop = threading.Event()
resource_samples = []


def resource_monitor():
    process = psutil.Process()

    process.cpu_percent(interval=None)
    psutil.cpu_percent(interval=None)

    while not monitor_stop.is_set():
        try:
            children = process.children(recursive=True)

            rss_bytes = process.memory_info().rss

            for child in children:
                try:
                    rss_bytes += child.memory_info().rss
                except (
                    psutil.NoSuchProcess,
                    psutil.AccessDenied,
                ):
                    pass

            disk = psutil.disk_io_counters()

            resource_samples.append(
                {{
                    "timestamp": time.time(),
                    "system_cpu_percent":
                        psutil.cpu_percent(interval=None),
                    "process_tree_rss_bytes": rss_bytes,
                    "system_memory_percent":
                        psutil.virtual_memory().percent,
                    "disk_read_bytes":
                        disk.read_bytes if disk else 0,
                    "disk_write_bytes":
                        disk.write_bytes if disk else 0,
                }}
            )

        except Exception:
            pass

        monitor_stop.wait(0.5)


def print_section(title):
    print()
    print("=" * 100)
    print(title)
    print("=" * 100)


try:
    print_section("STARTING APACHE SEDONA CATALOG JOB")

    config = (
        SedonaContext.builder()
        .master("local[2]")
        .appName("GHSL-Sedona-Metadata-Catalog")
        .config("spark.driver.memory", "3g")
        .config("spark.sql.shuffle.partitions", "4")
        .config("spark.default.parallelism", "2")
        .config("spark.ui.enabled", "false")
        .config("spark.sql.adaptive.enabled", "true")
        .getOrCreate()
    )

    spark = SedonaContext.create(config)
    spark.sparkContext.setLogLevel("WARN")

    print("Spark version:", spark.version)
    print("Spark master:", spark.sparkContext.master)
    print(
        "Default parallelism:",
        spark.sparkContext.defaultParallelism,
    )
    print("Raw GHSL root:", RAW_ROOT)
    print("Catalog output:", CATALOG_PATH)
    print("Benchmark output:", BENCHMARK_PATH)

    monitor_thread = threading.Thread(
        target=resource_monitor,
        daemon=True,
    )
    monitor_thread.start()

    benchmark_start_utc = datetime.now(
        timezone.utc
    ).isoformat()

    total_start = time.perf_counter()

    # --------------------------------------------------------
    # Read all GeoTIFF metadata through Apache Sedona
    # --------------------------------------------------------

    print_section("READING ALL GHSL GEOTIFF HEADERS")

    read_start = time.perf_counter()

    metadata = (
        spark.read
        .format("geotiff.metadata")
        .option("recursiveFileLookup", "true")
        .option("pathGlobFilter", "*.tif")
        .load(RAW_ROOT)
    )

    metadata = metadata.persist()

    metadata_count = metadata.count()

    read_seconds = time.perf_counter() - read_start

    print("Metadata rows:", metadata_count)
    print("Header scan time:", round(read_seconds, 3), "seconds")

    if metadata_count != EXPECTED_FILES:
        raise RuntimeError(
            f"Expected {{EXPECTED_FILES}} metadata rows, "
            f"but Sedona returned {{metadata_count}}."
        )

    # --------------------------------------------------------
    # Derive project control columns
    # --------------------------------------------------------

    print_section("DERIVING PROJECT CATALOG FIELDS")

    lower_path = F.lower(F.col("path"))

    product = (
        F.when(
            lower_path.contains("building_height"),
            F.lit("building_height"),
        )
        .when(
            lower_path.contains("built_surface"),
            F.lit("built_surface"),
        )
        .when(
            lower_path.contains("population"),
            F.lit("population"),
        )
        .when(
            lower_path.contains("smod"),
            F.lit("smod"),
        )
        .otherwise(F.lit("unknown"))
    )

    temporal_value = F.regexp_extract(
        F.col("path"),
        r"(?:epoch|year)=([0-9]{{4}})",
        1,
    ).cast("integer")

    catalog = (
        metadata
        .withColumn(
            "filename",
            F.regexp_extract(
                F.col("path"),
                r"([^/]+)$",
                1,
            ),
        )
        .withColumn("product", product)
        .withColumn("temporal_value", temporal_value)
        .withColumn(
            "temporal_type",
            F.when(
                F.col("product") == "building_height",
                F.lit("year"),
            ).otherwise(F.lit("epoch")),
        )
        .withColumn(
            "resolution_x",
            F.abs(F.col("geoTransform.scaleX")),
        )
        .withColumn(
            "resolution_y",
            F.abs(F.col("geoTransform.scaleY")),
        )
        .withColumn(
            "min_x",
            F.col("cornerCoordinates.minX"),
        )
        .withColumn(
            "min_y",
            F.col("cornerCoordinates.minY"),
        )
        .withColumn(
            "max_x",
            F.col("cornerCoordinates.maxX"),
        )
        .withColumn(
            "max_y",
            F.col("cornerCoordinates.maxY"),
        )
        .withColumn(
            "pixel_count",
            F.col("width").cast("long")
            * F.col("height").cast("long"),
        )
        .withColumn(
            "overview_count",
            F.size(F.col("overviews")),
        )
        .withColumn(
            "band_data_types",
            F.transform(
                F.col("bands"),
                lambda band: band["dataType"],
            ),
        )
        .withColumn(
            "band_nodata_values",
            F.transform(
                F.col("bands"),
                lambda band: band["noDataValue"],
            ),
        )
        .withColumn(
            "catalog_created_utc",
            F.lit(
                datetime.now(timezone.utc).isoformat()
            ),
        )
        .select(
            "product",
            "temporal_type",
            "temporal_value",
            "filename",
            "path",
            "driver",
            "fileSize",
            "width",
            "height",
            "pixel_count",
            "numBands",
            "band_data_types",
            "band_nodata_values",
            "srid",
            "crs",
            "resolution_x",
            "resolution_y",
            "min_x",
            "min_y",
            "max_x",
            "max_y",
            "geoTransform",
            "cornerCoordinates",
            "bands",
            "overviews",
            "overview_count",
            "isTiled",
            "compression",
            "metadata",
            "catalog_created_utc",
        )
    )

    catalog = catalog.persist()

    catalog_count = catalog.count()

    if catalog_count != EXPECTED_FILES:
        raise RuntimeError(
            f"Derived catalog contains {{catalog_count}} rows; "
            f"expected {{EXPECTED_FILES}}."
        )

    unknown_products = (
        catalog
        .filter(F.col("product") == "unknown")
        .count()
    )

    missing_temporal_values = (
        catalog
        .filter(F.col("temporal_value").isNull())
        .count()
    )

    if unknown_products != 0:
        raise RuntimeError(
            f"Catalog contains {{unknown_products}} "
            "unclassified product rows."
        )

    if missing_temporal_values != 0:
        raise RuntimeError(
            f"Catalog contains {{missing_temporal_values}} "
            "rows without an epoch/year."
        )

    # --------------------------------------------------------
    # Quantitative validation
    # --------------------------------------------------------

    print_section("CATALOG VALIDATION SUMMARY")

    total_file_bytes = (
        catalog
        .agg(F.sum("fileSize").alias("total_bytes"))
        .first()["total_bytes"]
    )

    if total_file_bytes != EXPECTED_BYTES:
        raise RuntimeError(
            f"Catalog byte total is {{total_file_bytes}}, "
            f"but expected {{EXPECTED_BYTES}}."
        )

    product_summary = (
        catalog
        .groupBy("product")
        .agg(
            F.count("*").alias("files"),
            F.sum("fileSize").alias("bytes"),
            F.min("temporal_value").alias("first_time"),
            F.max("temporal_value").alias("last_time"),
            F.collect_set("resolution_x").alias(
                "x_resolutions"
            ),
            F.collect_set("resolution_y").alias(
                "y_resolutions"
            ),
            F.collect_set("numBands").alias(
                "band_counts"
            ),
            F.collect_set("compression").alias(
                "compressions"
            ),
        )
        .orderBy("product")
    )

    product_summary.show(
        n=20,
        truncate=False,
    )

    expected_product_counts = {{
        "building_height": 15,
        "built_surface": 150,
        "population": 150,
        "smod": 10,
    }}

    actual_product_counts = {{
        row["product"]: row["files"]
        for row in product_summary.collect()
    }}

    if actual_product_counts != expected_product_counts:
        raise RuntimeError(
            "Unexpected product counts. "
            f"Expected {{expected_product_counts}}, "
            f"found {{actual_product_counts}}."
        )

    duplicate_paths = (
        catalog
        .groupBy("path")
        .count()
        .filter(F.col("count") != 1)
        .count()
    )

    if duplicate_paths != 0:
        raise RuntimeError(
            f"Catalog contains {{duplicate_paths}} "
            "duplicate paths."
        )

    # --------------------------------------------------------
    # Write catalog to HDFS Parquet
    # --------------------------------------------------------

    print_section("WRITING CURATED PARQUET CATALOG")

    write_start = time.perf_counter()

    (
        catalog
        .repartition(
            4,
            "product",
            "temporal_value",
        )
        .write
        .mode("overwrite")
        .partitionBy(
            "product",
            "temporal_type",
        )
        .parquet(CATALOG_PATH)
    )

    write_seconds = time.perf_counter() - write_start

    print("Catalog write time:", round(write_seconds, 3), "seconds")

    # --------------------------------------------------------
    # Read catalog back from HDFS and validate
    # --------------------------------------------------------

    print_section("READ-BACK VALIDATION")

    validation_start = time.perf_counter()

    saved_catalog = spark.read.parquet(CATALOG_PATH)

    saved_count = saved_catalog.count()

    saved_bytes = (
        saved_catalog
        .agg(F.sum("fileSize").alias("total_bytes"))
        .first()["total_bytes"]
    )

    validation_seconds = (
        time.perf_counter() - validation_start
    )

    print("Saved catalog rows:", saved_count)
    print("Saved source-byte total:", saved_bytes)
    print(
        "Read-back validation time:",
        round(validation_seconds, 3),
        "seconds",
    )

    if saved_count != EXPECTED_FILES:
        raise RuntimeError(
            f"Saved catalog contains {{saved_count}} rows; "
            f"expected {{EXPECTED_FILES}}."
        )

    if saved_bytes != EXPECTED_BYTES:
        raise RuntimeError(
            f"Saved catalog byte total is {{saved_bytes}}; "
            f"expected {{EXPECTED_BYTES}}."
        )

    # --------------------------------------------------------
    # Finish resource monitoring and compute metrics
    # --------------------------------------------------------

    total_seconds = time.perf_counter() - total_start

    monitor_stop.set()
    monitor_thread.join(timeout=3)

    cpu_values = [
        item["system_cpu_percent"]
        for item in resource_samples
    ]

    rss_values = [
        item["process_tree_rss_bytes"]
        for item in resource_samples
    ]

    memory_percent_values = [
        item["system_memory_percent"]
        for item in resource_samples
    ]

    disk_read_delta = 0
    disk_write_delta = 0

    if len(resource_samples) >= 2:
        disk_read_delta = max(
            0,
            resource_samples[-1]["disk_read_bytes"]
            - resource_samples[0]["disk_read_bytes"],
        )

        disk_write_delta = max(
            0,
            resource_samples[-1]["disk_write_bytes"]
            - resource_samples[0]["disk_write_bytes"],
        )

    input_mib = EXPECTED_BYTES / (1024 ** 2)

    metrics = {{
        "benchmark_name":
            "sedona_geotiff_metadata_catalog",
        "benchmark_start_utc":
            benchmark_start_utc,
        "benchmark_end_utc":
            datetime.now(timezone.utc).isoformat(),
        "engine":
            "Apache Sedona geotiff.metadata",
        "spark_version":
            spark.version,
        "spark_master":
            spark.sparkContext.master,
        "default_parallelism":
            spark.sparkContext.defaultParallelism,
        "host":
            platform.node(),
        "python_version":
            platform.python_version(),
        "cpu_logical_count":
            psutil.cpu_count(logical=True),
        "system_memory_bytes":
            psutil.virtual_memory().total,
        "input_path":
            RAW_ROOT,
        "output_path":
            CATALOG_PATH,
        "input_files":
            EXPECTED_FILES,
        "input_bytes":
            EXPECTED_BYTES,
        "input_mib":
            input_mib,
        "catalog_rows":
            saved_count,
        "header_scan_seconds":
            read_seconds,
        "catalog_write_seconds":
            write_seconds,
        "readback_validation_seconds":
            validation_seconds,
        "total_seconds":
            total_seconds,
        "files_per_second":
            EXPECTED_FILES / total_seconds,
        "mib_per_second":
            input_mib / total_seconds,
        "mean_system_cpu_percent":
            statistics.mean(cpu_values)
            if cpu_values else None,
        "peak_system_cpu_percent":
            max(cpu_values)
            if cpu_values else None,
        "peak_process_tree_rss_bytes":
            max(rss_values)
            if rss_values else None,
        "peak_system_memory_percent":
            max(memory_percent_values)
            if memory_percent_values else None,
        "disk_read_bytes_delta":
            disk_read_delta,
        "disk_write_bytes_delta":
            disk_write_delta,
        "resource_sample_count":
            len(resource_samples),
        "source_file_count_validation":
            "PASS",
        "source_byte_validation":
            "PASS",
        "product_count_validation":
            "PASS",
        "duplicate_path_validation":
            "PASS",
        "status":
            "SUCCESS",
    }}

    print_section("QUANTITATIVE PERFORMANCE METRICS")

    for key, value in metrics.items():
        print(f"{{key}}: {{value}}")

    benchmark_df = spark.createDataFrame([metrics])

    (
        benchmark_df
        .coalesce(1)
        .write
        .mode("overwrite")
        .parquet(BENCHMARK_PATH)
    )

    # --------------------------------------------------------
    # Final checkpoint
    # --------------------------------------------------------

    print_section("FINAL RESULT")

    print("Raw GeoTIFF files cataloged:", saved_count)
    print("Raw source bytes verified:", saved_bytes)
    print("Catalog HDFS path:", CATALOG_PATH)
    print("Benchmark HDFS path:", BENCHMARK_PATH)
    print("Catalog status: VALIDATED")
    print("SEDONA GHSL CATALOG BUILD: SUCCESS")

finally:
    monitor_stop.set()

    if spark is not None:
        try:
            spark.stop()
        except Exception:
            pass
''').strip() + "\n"

LOCAL_SCRIPT.write_text(
    script_content,
    encoding="utf-8",
)

print("Local catalog script created:", LOCAL_SCRIPT)
print("Script size:", LOCAL_SCRIPT.stat().st_size, "bytes")

# ------------------------------------------------------------
# 2. Copy the job to AWS
# ------------------------------------------------------------

scp_result = subprocess.run(
    [
        "scp",
        "-o", "StrictHostKeyChecking=no",
        "-o", "UserKnownHostsFile=/dev/null",
        "-o", "BatchMode=yes",
        "-o", "ConnectTimeout=30",
        "-i", SSH_KEY,
        str(LOCAL_SCRIPT),
        f"{AWS_USER}@{AWS_HOST}:{REMOTE_SCRIPT}",
    ],
    text=True,
    capture_output=True,
)

if scp_result.returncode != 0:
    print(scp_result.stderr)
    raise RuntimeError(
        "Failed to copy the Sedona catalog job to AWS."
    )

print("Catalog job copied to AWS successfully.")

# ------------------------------------------------------------
# 3. Execute with cached Sedona dependencies
# ------------------------------------------------------------

run_command = (
    "source /home/ubuntu/ghsl-sedona-env.sh && "
    "spark-submit "
    "--master local[2] "
    "--packages "
    "org.apache.sedona:sedona-spark-3.5_2.12:1.9.1,"
    "org.datasyslab:geotools-wrapper:1.9.1-33.5 "
    "--repositories "
    "https://artifacts.unidata.ucar.edu/repository/unidata-all "
    f"{REMOTE_SCRIPT}"
)

result = subprocess.run(
    ["bash", REMOTE_EXEC, run_command],
    text=True,
    capture_output=True,
    timeout=1800,
)

print(result.stdout)

if result.stderr:
    print("REMOTE STDERR:")
    print(result.stderr)

print("Return code:", result.returncode)

if result.returncode != 0:
    raise RuntimeError(
        "Sedona GHSL catalog build failed with "
        f"return code {result.returncode}."
    )

GHSL INDIA GEOSPATIAL DATA PIPELINE - CURRENT PROJECT SUMMARY

1. AWS INFRASTRUCTURE

The EC2 instance was upgraded to m7i-flex.large.

Instance configuration:
- Operating system: Ubuntu 26.04 LTS
- Architecture: x86_64
- CPU: 2 logical vCPUs
- Memory: approximately 7.6 GiB
- Disk: approximately 77 GiB
- Java: OpenJDK 11.0.32
- Public IP address: 34.207.116.148

The larger instance provides enough memory to run HDFS, Spark, and Apache Sedona together without adding swap at this stage.


2. HDFS STATUS

Hadoop version:
- Hadoop 3.4.1

Running HDFS services:
- NameNode
- DataNode
- SecondaryNameNode

HDFS endpoint:
- hdfs://localhost:9000

HDFS status:
- Safe mode: OFF
- Filesystem health: HEALTHY
- Missing blocks: 0
- Missing replicas: 0
- HDFS is writable

GHSL raw-zone location:
- /data/ghsl/raw/india

Raw GHSL inventory:
- Total GeoTIFF files: 325
- Total logical size: 7,038,596,744 bytes
- Approximate size: 6.6 GiB
- Total validated HDFS blocks: 325

A temporary HDFS file was successfully written, read, and deleted. This confirmed that HDFS is fully operational and writable.


3. PYTHON ENVIRONMENT

Ubuntu system Python remains unchanged:
- System Python: Python 3.14.4
- Path: /usr/bin/python3

A separate project environment was created:
- Project Python: Python 3.12.16
- Environment path: /home/ubuntu/ghsl-sedona-env
- Configuration file: /home/ubuntu/ghsl-sedona-env.sh

The isolated environment does not replace or modify Ubuntu's system Python and does not affect HDFS.

Installed Python packages:
- PySpark 3.5.9
- Apache Sedona 1.9.1
- Rasterio 1.5.2
- NumPy 2.5.4
- Pandas 3.1.0
- PyArrow 26.0.0
- psutil 7.2.2
- Shapely 2.2.0


4. SPARK AND SEDONA CONFIGURATION

Spark configuration:
- Spark version: 3.5.9
- Scala version: 2.12.18
- Java version: 11.0.32
- Execution mode: local[2]
- Default parallelism: 2

Apache Sedona configuration:
- Apache Sedona version: 1.9.1
- Sedona Spark artifact:
  org.apache.sedona:sedona-spark-3.5_2.12:1.9.1
- GeoTools wrapper:
  org.datasyslab:geotools-wrapper:1.9.1-33.5

The required Sedona and GeoTools JAR dependencies were successfully downloaded and cached under:
- /home/ubuntu/.ivy2

Spark and Sedona can therefore be started without downloading the complete dependency set again.


5. SEDONA SPATIAL SQL TEST

Apache Sedona was successfully initialized inside Spark.

The spatial SQL smoke test created a point at:
- X coordinate: 77.5946
- Y coordinate: 12.9716

Sedona returned:
- POINT (77.5946 12.9716)

A containment operation checked whether this point was inside a specified spatial envelope.

Result:
- Point inside envelope: true

Final status:
- Sedona spatial SQL: SUCCESS


6. SPARK-TO-HDFS CONNECTIVITY TEST

Spark successfully connected to:
- hdfs://localhost:9000

Spark verified the GHSL raw-zone directory:
- /data/ghsl/raw/india

Spark observed:
- HDFS path exists: true
- File count: 325
- Logical bytes: 7,038,596,744

Final status:
- Spark-to-HDFS connectivity: SUCCESS


7. REAL GHSL GEOTIFF TEST

Apache Sedona successfully read metadata from a real GHSL GeoTIFF directly from HDFS.

Test file:
- /data/ghsl/raw/india/smod/epoch=1975/ghsl_india_smod_1975_1000m_r2023a.tif

Complete HDFS URI:
- hdfs://localhost:9000/data/ghsl/raw/india/smod/epoch=1975/ghsl_india_smod_1975_1000m_r2023a.tif

Extracted metadata:
- Driver: GTiff
- File size: 1,071,124 bytes
- Width: 2,902 pixels
- Height: 3,179 pixels
- Number of bands: 1
- Band data type: Signed 16-bit integer
- Color interpretation: Gray
- Pixel resolution: 1,000 m x 1,000 m
- Projection: World Mollweide
- Internal tiling: Yes
- Tile size: 256 x 256 pixels
- Compression: LZW
- Overview levels: 5
- NoData value: Not explicitly recorded

Affine transformation:
- Upper-left X: 6,471,000
- Upper-left Y: 4,013,000
- X resolution: 1,000
- Y resolution: -1,000
- X skew: 0
- Y skew: 0

Raster bounds:
- Minimum X: 6,471,000
- Minimum Y: 834,000
- Maximum X: 9,373,000
- Maximum Y: 4,013,000

The dimensions and bounds are internally consistent:

Width:
(9,373,000 - 6,471,000) / 1,000 = 2,902 pixels

Height:
(4,013,000 - 834,000) / 1,000 = 3,179 pixels

The negative Y resolution is correct for a north-up raster because row numbers increase downward while projected northing decreases.

Sedona reported SRID 0 but successfully extracted the complete World Mollweide CRS definition. SRID 0 means Sedona did not map the WKT to a standard EPSG integer. It does not mean the raster is missing its coordinate system.

Project CRS interpretation:
- Canonical project CRS: ESRI:54009
- Sedona metadata SRID: 0
- Sedona stored CRS: World Mollweide WKT

Final status:
- Real GHSL GeoTIFF metadata read: SUCCESS


8. COMPLETE GHSL METADATA CATALOG

A complete Apache Sedona metadata catalog was created for all 325 raw GeoTIFF files.

Catalog output:
- hdfs://localhost:9000/data/ghsl/metadata/india/geotiff_catalog

Benchmark output:
- hdfs://localhost:9000/data/ghsl/metadata/india/catalog_benchmark

Output format:
- Apache Parquet

The catalog contains one row for every physical GeoTIFF file and includes:
- Product name
- Epoch or year
- Filename
- HDFS path
- File size
- Width
- Height
- Pixel count
- Band count
- Band data types
- NoData values
- CRS
- Resolution
- Spatial bounds
- Affine transformation
- Internal tiling information
- Compression
- Overview information
- Source metadata
- Catalog creation timestamp


9. PRODUCT INVENTORY

Building height:
- Files: 15
- Total size: 391,753,971 bytes
- Temporal value: 2018
- Resolution: 100 m
- Bands: 1
- Compression: LZW

Built surface:
- Files: 150
- Total size: 1,244,708,193 bytes
- Temporal coverage: 1975-2020
- Resolution: 100 m
- Bands: 2
- Compression: LZW

Population:
- Files: 150
- Total size: 5,390,206,686 bytes
- Temporal coverage: 1975-2020
- Resolution: 100 m
- Bands: 1
- Compression: LZW

SMOD:
- Files: 10
- Total size: 11,927,894 bytes
- Temporal coverage: 1975-2020
- Resolution: 1,000 m
- Bands: 1
- Compression: LZW

Complete inventory:
- Building height: 15 files
- Built surface: 150 files
- Population: 150 files
- SMOD: 10 files
- Total: 325 files

The complete byte total is:
- 7,038,596,744 bytes


10. CATALOG VALIDATION

The catalog was validated before and after being written to HDFS.

Validation results:
- Source file count: PASS
- Source byte count: PASS
- Product classification: PASS
- Product file counts: PASS
- Temporal-value extraction: PASS
- Duplicate-path check: PASS
- Saved Parquet row count: PASS
- Saved Parquet byte-total check: PASS

Read-back results:
- Saved catalog rows: 325
- Saved source-byte total: 7,038,596,744

Final catalog status:
- VALIDATED
- SUCCESS


11. CATALOG BENCHMARK

Benchmark environment:
- Spark version: 3.5.9
- Python version: 3.12.16
- Spark execution mode: local[2]
- Logical CPU count: 2
- System memory: 8,153,108,480 bytes
- Default Spark parallelism: 2

Benchmark workload:
- Input files: 325
- Logical input size: 7,038,596,744 bytes
- Logical input size: approximately 6,712.53 MiB
- Catalog rows produced: 325

Timing results:
- GeoTIFF header scan: 23.982 seconds
- Parquet catalog write: 4.606 seconds
- Read-back validation: 1.215 seconds
- Total end-to-end runtime: 44.649 seconds

Reported processing rates:
- Files per second: 7.279
- Logical dataset MiB per second: 150.341

Resource use:
- Mean system CPU utilization: 95.34%
- Peak system CPU utilization: 100%
- Peak process-tree RSS: 249,606,144 bytes
- Peak total system memory utilization: 32.1%
- Disk-read delta: 623,374,336 bytes
- Disk-write delta: 3,665,920 bytes
- Resource-monitoring samples: 88

The benchmark indicates that the two-vCPU instance was CPU-saturated during the catalog operation. Memory remained comfortable, so this workload was CPU-limited rather than memory-limited.


12. BENCHMARK INTERPRETATION

The reported 150.341 MiB/s value represents:

Total logical size of the cataloged GeoTIFF files divided by the complete job runtime.

It should be described as:
- Logical dataset cataloging rate

It should not be described as:
- Physical disk-read throughput

Apache Sedona's metadata reader reads GeoTIFF headers and metadata structures without decoding every raster pixel. The measured physical system disk-read delta was approximately 623.4 MB, or about 594.5 MiB.

Therefore, the benchmark should report these separately:

- Logical dataset cataloging rate: 150.34 MiB/s
- Files cataloged per second: 7.28
- Measured disk reads during execution: approximately 594.5 MiB
- End-to-end runtime: 44.65 seconds

This distinction makes the benchmark technically and academically defensible.


13. CURRENT PROJECT STATUS

Completed tasks:
- AWS EC2 instance upgrade
- HDFS startup and health verification
- HDFS write-readiness validation
- Verification of all 325 GHSL files
- Isolated Python 3.12 environment installation
- PySpark 3.5.9 installation
- Apache Sedona 1.9.1 installation
- Compatible Scala and Sedona JAR configuration
- Sedona spatial SQL smoke test
- Spark-to-HDFS connectivity test
- Real GHSL GeoTIFF metadata test
- Complete 325-file metadata catalog
- Product and temporal classification
- Catalog storage as Parquet in HDFS
- Catalog read-back validation
- Quantitative catalog benchmark
- CPU, memory, and disk monitoring

Current infrastructure status:
- HDFS: Healthy and writable
- Spark: Operational
- Apache Sedona: Operational
- Spark-to-HDFS integration: Verified
- Raw GHSL dataset: Intact
- Metadata catalog: Validated
- Benchmark record: Saved in HDFS


14. NEXT RECOMMENDED TASK

The next stage should move from metadata processing to actual raster-pixel processing.

Recommended first workload:
- Select one 100 m GHSL population shard
- Load actual raster pixel values
- Inspect raster dimensions and tiling
- Determine valid and invalid pixels
- Calculate minimum, maximum, mean, sum, and valid-pixel count
- Record execution time and resource consumption
- Compare a serial Rasterio implementation with a Spark/Sedona implementation

This will provide a true computational benchmark involving pixel decoding and raster analytics, rather than only GeoTIFF header indexing.

In [ ]:
# ============================================================
# Corrected pixel-level benchmark
# Handles embedded NaN values when GeoTIFF NoData is undefined
# ============================================================

from pathlib import Path
import subprocess
import textwrap

REMOTE_EXEC = "/content/drive/MyDrive/remote_exec.sh"
SSH_KEY = "/content/hadoop-hdfs.pem"

AWS_HOST = "54.92.192.154"
AWS_USER = "ubuntu"

LOCAL_SCRIPT = Path(
    "/content/ghsl_population_pixel_benchmark_fixed.py"
)

REMOTE_SCRIPT = (
    "/home/ubuntu/ghsl_population_pixel_benchmark_fixed.py"
)

# ------------------------------------------------------------
# 1. Select one 1975 population shard
# ------------------------------------------------------------

find_command = (
    "export HADOOP_HOME=/usr/local/hadoop; "
    "export PATH=$HADOOP_HOME/bin:$HADOOP_HOME/sbin:$PATH; "
    "hdfs dfs -ls -R "
    "/data/ghsl/raw/india/population/epoch=1975 "
    "| awk '$1 ~ /^-/ && tolower($NF) ~ /\\.tiff?$/ "
    "{print $5, $NF}' "
    "| sort -n "
    "| head -1"
)

find_result = subprocess.run(
    ["bash", REMOTE_EXEC, find_command],
    text=True,
    capture_output=True,
)

if find_result.returncode != 0:
    print(find_result.stderr)
    raise RuntimeError(
        "Could not locate a 1975 population GeoTIFF."
    )

selected_line = find_result.stdout.strip()

if not selected_line:
    raise RuntimeError(
        "No 1975 population GeoTIFF was found."
    )

size_text, hdfs_path = selected_line.split(maxsplit=1)

HDFS_PATH = hdfs_path
HDFS_URI = f"hdfs://localhost:9000{HDFS_PATH}"
SOURCE_BYTES = int(size_text)

print("Selected population shard:")
print(HDFS_PATH)
print("Source size:", SOURCE_BYTES, "bytes")
print("Source URI:", HDFS_URI)

# ------------------------------------------------------------
# 2. Create corrected benchmark script
# ------------------------------------------------------------

script_content = textwrap.dedent(f'''
import gc
import math
import os
import platform
import subprocess
import time
from datetime import datetime, timezone

import numpy as np
import psutil
import rasterio

from pyspark.sql.functions import col, udf
from pyspark.sql.types import (
    DoubleType,
    LongType,
    StructField,
    StructType,
)

from sedona.spark import SedonaContext


HDFS_PATH = {HDFS_PATH!r}
HDFS_URI = {HDFS_URI!r}
SOURCE_BYTES = {SOURCE_BYTES}

LOCAL_TIF = "/tmp/ghsl_population_benchmark_fixed.tif"

RESULT_PATH = (
    "hdfs://localhost:9000/data/ghsl/benchmarks/"
    "india/population_pixel_benchmark"
)

spark = None


def section(title):
    print()
    print("=" * 100)
    print(title)
    print("=" * 100)


def close_enough(
    left,
    right,
    relative=1e-7,
    absolute=1e-6,
):
    if left is None or right is None:
        return left is None and right is None

    return math.isclose(
        float(left),
        float(right),
        rel_tol=relative,
        abs_tol=absolute,
    )


statistics_schema = StructType(
    [
        StructField("total_count", LongType(), False),
        StructField("valid_count", LongType(), False),
        StructField("invalid_count", LongType(), False),
        StructField("value_sum", DoubleType(), False),
        StructField("value_mean", DoubleType(), False),
        StructField("value_stddev", DoubleType(), False),
        StructField("value_min", DoubleType(), False),
        StructField("value_max", DoubleType(), False),
    ]
)


@udf(returnType=statistics_schema)
def finite_statistics_udf(raster):
    # SedonaRaster.as_numpy() returns:
    # (bands, height, width)
    values = raster.as_numpy()[0]

    total_count = int(values.size)

    # Critical correction:
    # The source declares no NoData value, but stores NaNs.
    finite_mask = np.isfinite(values)
    finite_values = values[finite_mask]

    valid_count = int(finite_values.size)
    invalid_count = int(total_count - valid_count)

    if valid_count == 0:
        raise ValueError(
            "The raster contains no finite pixel values."
        )

    finite_values_64 = finite_values.astype(
        np.float64,
        copy=False,
    )

    return (
        total_count,
        valid_count,
        invalid_count,
        float(np.sum(finite_values_64)),
        float(np.mean(finite_values_64)),
        float(np.std(finite_values_64, ddof=0)),
        float(np.min(finite_values_64)),
        float(np.max(finite_values_64)),
    )


try:
    benchmark_started_utc = datetime.now(
        timezone.utc
    ).isoformat()

    section("SOURCE FILE")

    print("HDFS path:", HDFS_PATH)
    print("HDFS URI:", HDFS_URI)
    print("Source bytes:", SOURCE_BYTES)

    # --------------------------------------------------------
    # Stage source for serial Rasterio baseline
    # --------------------------------------------------------

    section("STAGING HDFS FILE FOR RASTERIO")

    if os.path.exists(LOCAL_TIF):
        os.remove(LOCAL_TIF)

    staging_start = time.perf_counter()

    subprocess.run(
        [
            "/usr/local/hadoop/bin/hdfs",
            "dfs",
            "-copyToLocal",
            "-f",
            HDFS_PATH,
            LOCAL_TIF,
        ],
        check=True,
    )

    staging_seconds = (
        time.perf_counter() - staging_start
    )

    local_size = os.path.getsize(LOCAL_TIF)

    print("Local path:", LOCAL_TIF)
    print("Local bytes:", local_size)
    print(
        "HDFS-to-local staging time:",
        round(staging_seconds, 6),
        "seconds",
    )

    if local_size != SOURCE_BYTES:
        raise RuntimeError(
            f"Staged size {{local_size}} does not match "
            f"HDFS size {{SOURCE_BYTES}}."
        )

    # --------------------------------------------------------
    # Rasterio finite-value baseline
    # --------------------------------------------------------

    section("RASTERIO FINITE-VALUE BASELINE")

    gc.collect()

    process = psutil.Process()

    rss_before = process.memory_info().rss
    cpu_before = process.cpu_times()

    rasterio_start = time.perf_counter()

    with rasterio.open(LOCAL_TIF) as src:
        rasterio_width = src.width
        rasterio_height = src.height
        rasterio_band_count = src.count
        rasterio_dtype = src.dtypes[0]
        rasterio_nodata = src.nodata
        rasterio_crs = str(src.crs) if src.crs else None

        # Read raw values because the GeoTIFF contains NaNs
        # without declaring a formal NoData value.
        values = src.read(1)

        rasterio_total_count = int(values.size)

        finite_mask = np.isfinite(values)
        finite_values = values[finite_mask]

        rasterio_valid_count = int(
            finite_values.size
        )

        rasterio_invalid_count = int(
            rasterio_total_count
            - rasterio_valid_count
        )

        if rasterio_valid_count == 0:
            raise RuntimeError(
                "Rasterio found no finite values."
            )

        finite_values_64 = finite_values.astype(
            np.float64,
            copy=False,
        )

        rasterio_sum = float(
            np.sum(finite_values_64)
        )

        rasterio_mean = float(
            np.mean(finite_values_64)
        )

        rasterio_stddev = float(
            np.std(finite_values_64, ddof=0)
        )

        rasterio_min = float(
            np.min(finite_values_64)
        )

        rasterio_max = float(
            np.max(finite_values_64)
        )

    rasterio_seconds = (
        time.perf_counter() - rasterio_start
    )

    cpu_after = process.cpu_times()
    rss_after = process.memory_info().rss

    rasterio_cpu_seconds = (
        cpu_after.user
        + cpu_after.system
        - cpu_before.user
        - cpu_before.system
    )

    print("Width:", rasterio_width)
    print("Height:", rasterio_height)
    print("Bands:", rasterio_band_count)
    print("Data type:", rasterio_dtype)
    print("CRS:", rasterio_crs)
    print("Declared NoData:", rasterio_nodata)
    print("Total pixels:", rasterio_total_count)
    print("Finite pixels:", rasterio_valid_count)
    print("Non-finite pixels:", rasterio_invalid_count)
    print("Sum:", rasterio_sum)
    print("Mean:", rasterio_mean)
    print("Population standard deviation:", rasterio_stddev)
    print("Minimum:", rasterio_min)
    print("Maximum:", rasterio_max)
    print(
        "Rasterio runtime:",
        round(rasterio_seconds, 6),
        "seconds",
    )
    print(
        "Rasterio CPU time:",
        round(rasterio_cpu_seconds, 6),
        "seconds",
    )
    print(
        "Rasterio RSS delta:",
        rss_after - rss_before,
        "bytes",
    )

    del values
    del finite_mask
    del finite_values
    del finite_values_64
    gc.collect()

    # --------------------------------------------------------
    # Start Spark and Sedona
    # --------------------------------------------------------

    section("STARTING APACHE SEDONA")

    config = (
        SedonaContext.builder()
        .master("local[2]")
        .appName(
            "GHSL-Population-Pixel-Benchmark-Fixed"
        )
        .config("spark.driver.memory", "3g")
        .config("spark.sql.shuffle.partitions", "2")
        .config("spark.default.parallelism", "2")
        .config("spark.ui.enabled", "false")
        .config("spark.sql.adaptive.enabled", "true")
        .getOrCreate()
    )

    spark = SedonaContext.create(config)
    spark.sparkContext.setLogLevel("WARN")

    print("Spark version:", spark.version)
    print("Spark master:", spark.sparkContext.master)
    print(
        "Default parallelism:",
        spark.sparkContext.defaultParallelism,
    )

    # --------------------------------------------------------
    # Sedona direct-HDFS finite-value statistics
    # --------------------------------------------------------

    section("SEDONA DIRECT-HDFS FINITE-VALUE STATISTICS")

    sedona_start = time.perf_counter()

    binary_df = (
        spark.read
        .format("binaryFile")
        .load(HDFS_URI)
    )

    raster_df = (
        binary_df
        .selectExpr(
            "path",
            "length",
            "RS_FromGeoTiff(content) AS raster",
        )
        .persist()
    )

    raster_rows = raster_df.count()

    if raster_rows != 1:
        raise RuntimeError(
            f"Expected one raster row, found "
            f"{{raster_rows}}."
        )

    sedona_dimensions = (
        raster_df
        .selectExpr(
            "length AS source_bytes",
            "RS_Width(raster) AS width",
            "RS_Height(raster) AS height",
            "RS_NumBands(raster) AS num_bands",
        )
        .first()
    )

    sedona_stats_row = (
        raster_df
        .select(
            finite_statistics_udf(
                col("raster")
            ).alias("stats")
        )
        .select("stats.*")
        .first()
    )

    sedona_seconds = (
        time.perf_counter() - sedona_start
    )

    sedona_width = int(
        sedona_dimensions["width"]
    )

    sedona_height = int(
        sedona_dimensions["height"]
    )

    sedona_band_count = int(
        sedona_dimensions["num_bands"]
    )

    sedona_total_count = int(
        sedona_stats_row["total_count"]
    )

    sedona_valid_count = int(
        sedona_stats_row["valid_count"]
    )

    sedona_invalid_count = int(
        sedona_stats_row["invalid_count"]
    )

    sedona_sum = float(
        sedona_stats_row["value_sum"]
    )

    sedona_mean = float(
        sedona_stats_row["value_mean"]
    )

    sedona_stddev = float(
        sedona_stats_row["value_stddev"]
    )

    sedona_min = float(
        sedona_stats_row["value_min"]
    )

    sedona_max = float(
        sedona_stats_row["value_max"]
    )

    print("Width:", sedona_width)
    print("Height:", sedona_height)
    print("Bands:", sedona_band_count)
    print("Total pixels:", sedona_total_count)
    print("Finite pixels:", sedona_valid_count)
    print("Non-finite pixels:", sedona_invalid_count)
    print("Sum:", sedona_sum)
    print("Mean:", sedona_mean)
    print("Population standard deviation:", sedona_stddev)
    print("Minimum:", sedona_min)
    print("Maximum:", sedona_max)
    print(
        "Sedona direct-HDFS runtime:",
        round(sedona_seconds, 6),
        "seconds",
    )

    # --------------------------------------------------------
    # Cross-engine validation
    # --------------------------------------------------------

    section("CROSS-ENGINE VALIDATION")

    checks = {{
        "Source byte count":
            int(
                sedona_dimensions["source_bytes"]
            ) == SOURCE_BYTES,

        "Raster width":
            sedona_width == rasterio_width,

        "Raster height":
            sedona_height == rasterio_height,

        "Band count":
            sedona_band_count
            == rasterio_band_count,

        "Total pixel count":
            sedona_total_count
            == rasterio_total_count,

        "Finite pixel count":
            sedona_valid_count
            == rasterio_valid_count,

        "Non-finite pixel count":
            sedona_invalid_count
            == rasterio_invalid_count,

        "Pixel-value sum":
            close_enough(
                sedona_sum,
                rasterio_sum,
                relative=1e-9,
                absolute=1e-3,
            ),

        "Pixel-value mean":
            close_enough(
                sedona_mean,
                rasterio_mean,
                relative=1e-9,
                absolute=1e-9,
            ),

        "Population standard deviation":
            close_enough(
                sedona_stddev,
                rasterio_stddev,
                relative=1e-8,
                absolute=1e-8,
            ),

        "Minimum pixel value":
            close_enough(
                sedona_min,
                rasterio_min,
            ),

        "Maximum pixel value":
            close_enough(
                sedona_max,
                rasterio_max,
            ),
    }}

    for name, passed in checks.items():
        print(
            f"{{name:<40}} "
            f"{{'PASS' if passed else 'FAIL'}}"
        )

    failed_checks = [
        name
        for name, passed in checks.items()
        if not passed
    ]

    if failed_checks:
        raise RuntimeError(
            "Cross-engine validation failed: "
            + ", ".join(failed_checks)
        )

    # --------------------------------------------------------
    # Save benchmark record
    # --------------------------------------------------------

    section("SAVING BENCHMARK RESULT")

    source_mib = SOURCE_BYTES / (1024 ** 2)

    result = {{
        "benchmark_name":
            "population_pixel_stats_finite_values",
        "benchmark_started_utc":
            benchmark_started_utc,
        "benchmark_finished_utc":
            datetime.now(timezone.utc).isoformat(),
        "host":
            platform.node(),
        "python_version":
            platform.python_version(),
        "spark_version":
            spark.version,
        "spark_master":
            spark.sparkContext.master,
        "logical_cpu_count":
            psutil.cpu_count(logical=True),
        "system_memory_bytes":
            psutil.virtual_memory().total,
        "source_hdfs_path":
            HDFS_PATH,
        "source_bytes":
            SOURCE_BYTES,
        "width":
            rasterio_width,
        "height":
            rasterio_height,
        "bands":
            rasterio_band_count,
        "declared_nodata":
            str(rasterio_nodata),
        "validity_rule":
            "numpy.isfinite",
        "total_pixels":
            rasterio_total_count,
        "finite_pixels":
            rasterio_valid_count,
        "nonfinite_pixels":
            rasterio_invalid_count,
        "pixel_sum":
            rasterio_sum,
        "pixel_mean":
            rasterio_mean,
        "pixel_stddev_population":
            rasterio_stddev,
        "pixel_min":
            rasterio_min,
        "pixel_max":
            rasterio_max,
        "hdfs_to_local_staging_seconds":
            staging_seconds,
        "rasterio_seconds":
            rasterio_seconds,
        "rasterio_cpu_seconds":
            rasterio_cpu_seconds,
        "rasterio_mib_per_second":
            source_mib / rasterio_seconds,
        "sedona_direct_hdfs_seconds":
            sedona_seconds,
        "sedona_direct_hdfs_mib_per_second":
            source_mib / sedona_seconds,
        "sedona_to_rasterio_time_ratio":
            sedona_seconds / rasterio_seconds,
        "cross_engine_validation":
            "PASS",
        "status":
            "SUCCESS",
    }}

    for key, value in result.items():
        print(f"{{key}}: {{value}}")

    (
        spark.createDataFrame([result])
        .coalesce(1)
        .write
        .mode("overwrite")
        .parquet(RESULT_PATH)
    )

    print("Benchmark saved to:", RESULT_PATH)

    section("FINAL RESULT")

    print("Embedded NaN handling: SUCCESS")
    print("Rasterio finite statistics: SUCCESS")
    print("Sedona finite statistics: SUCCESS")
    print("Cross-engine validation: PASS")
    print("Benchmark record saved in HDFS.")
    print("POPULATION PIXEL BENCHMARK: SUCCESS")

finally:
    if spark is not None:
        try:
            spark.stop()
        except Exception:
            pass

    if os.path.exists(LOCAL_TIF):
        try:
            os.remove(LOCAL_TIF)
        except Exception:
            pass
''').strip() + "\n"

LOCAL_SCRIPT.write_text(
    script_content,
    encoding="utf-8",
)

print()
print("Corrected script created:", LOCAL_SCRIPT)
print("Script size:", LOCAL_SCRIPT.stat().st_size, "bytes")

# ------------------------------------------------------------
# 3. Copy corrected job to AWS
# ------------------------------------------------------------

scp_result = subprocess.run(
    [
        "scp",
        "-o", "StrictHostKeyChecking=no",
        "-o", "UserKnownHostsFile=/dev/null",
        "-o", "BatchMode=yes",
        "-o", "ConnectTimeout=30",
        "-i", SSH_KEY,
        str(LOCAL_SCRIPT),
        f"{AWS_USER}@{AWS_HOST}:{REMOTE_SCRIPT}",
    ],
    text=True,
    capture_output=True,
)

if scp_result.returncode != 0:
    print(scp_result.stderr)
    raise RuntimeError(
        "Failed to copy corrected benchmark script."
    )

print("Corrected script copied to AWS.")

# ------------------------------------------------------------
# 4. Run corrected benchmark
# ------------------------------------------------------------

run_command = (
    "source /home/ubuntu/ghsl-sedona-env.sh && "
    "spark-submit "
    "--master local[2] "
    "--driver-memory 3g "
    "--packages "
    "org.apache.sedona:sedona-spark-3.5_2.12:1.9.1,"
    "org.datasyslab:geotools-wrapper:1.9.1-33.5 "
    "--repositories "
    "https://artifacts.unidata.ucar.edu/repository/unidata-all "
    f"{REMOTE_SCRIPT}"
)

result = subprocess.run(
    ["bash", REMOTE_EXEC, run_command],
    text=True,
    capture_output=True,
    timeout=1800,
)

print()
print(result.stdout)

if result.stderr:
    print("REMOTE STDERR:")
    print(result.stderr)

print("Return code:", result.returncode)

if result.returncode != 0:
    raise RuntimeError(
        "Corrected population benchmark failed with "
        f"return code {result.returncode}."
    )

Current checkpoint

Single population-shard pixel decoding              COMPLETE
Embedded NaN detection                              COMPLETE
Finite-value validity rule                          ESTABLISHED
Rasterio serial statistics                          COMPLETE
Sedona direct-HDFS statistics                       COMPLETE
Cross-engine numerical validation                   PASSED
Single-shard benchmark                              SAVED

What the current HDFS inventory means

At present, our metadata catalog showed:

Plain Text
Population: 150 files = 15 shards x 10 epochs
Built surface: 150 files = 15 shards x 10 epochs
SMOD: 10 files = 1 raster x 10 epochs


That strongly indicates that the currently downloaded raw data contains only:

Plain Text
1975-2020


The 2025 and 2030 products are not yet present in the existing catalog. Our next all-shard benchmark for 1975 is therefore correctly within the historical period.

Correct next sequence
Plain Text
Step 1:
Process all 15 population shards for 1975.

Step 2:
Validate the epoch-wide population total and benchmark parallel execution.

Step 3:
Generalize the same process across all ten historical epochs:
1975-2020.

Step 4:
Process built surface across the same historical epochs.

Step 5:
Aggregate population and built surface from 100 m to the aligned
1 km SMOD grid.

Step 6:
Join population, built surface, and SMOD into the grid-year dataset.

Step 7:
Download and ingest the 2025 and 2030 test epochs separately.

Step 8:
Keep 2025 and 2030 out of any calibration or rule-setting stage and
use them only for held-out evaluation.

In [11]:
from pathlib import Path
import subprocess
import textwrap


# ============================================================
# NOTEBOOK CONFIGURATION
# ============================================================

LOCAL_SCRIPT = Path(
    "/content/ghsl_population_1975_sedona_safe.py"
)

REMOTE_SCRIPT = (
    "/home/ubuntu/ghsl_population_1975_sedona_safe.py"
)

REMOTE_EXEC = "/content/drive/MyDrive/remote_exec.sh"
SSH_KEY = "/content/hadoop-hdfs.pem"

AWS_USER = "ubuntu"
AWS_HOST = "54.92.192.154"


# ============================================================
# CLEAN STANDALONE SPARK + SEDONA + HDFS PROGRAM
#
# Important design:
#
# 1. Spark reads the GeoTIFF inventory directly from HDFS.
# 2. Only one source GeoTIFF is processed at a time.
# 3. Sedona decodes the selected GeoTIFF in the JVM.
# 4. Sedona splits it into 256 x 256 raster tiles.
# 5. Sedona calculates scalar statistics per tile.
# 6. Spark aggregates the tile statistics.
# 7. Only one tiny scalar result row is collected to Python.
# 8. Final results are written to HDFS as Parquet.
#
# There is:
#
# - no Rasterio
# - no NumPy
# - no Python UDF
# - no RS_BandAsArray
# - no RS_CountValue
# - no raster cache
# - no raster checkpoint
# - no raster repartition/shuffle
# ============================================================

REMOTE_PROGRAM = r'''
import math
import sys
import time
import traceback

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    LongType,
    IntegerType,
    DoubleType,
)


# ============================================================
# CONSTANTS
# ============================================================

EPOCH = 1975
EXPECTED_SHARDS = 15

TILE_SIZE = 256
INVALID_SENTINEL = -200.0

HDFS_INPUT = (
    "hdfs://localhost:9000"
    "/data/ghsl/raw/india/population/epoch=1975"
)

HDFS_OUTPUT_BASE = (
    "hdfs://localhost:9000"
    "/data/ghsl/processed/india/population/epoch=1975"
)

HDFS_SHARD_OUTPUT = (
    HDFS_OUTPUT_BASE
    + "/sedona_shard_statistics"
)

HDFS_EPOCH_OUTPUT = (
    HDFS_OUTPUT_BASE
    + "/sedona_epoch_statistics"
)


# ============================================================
# FORMATTING HELPER
# ============================================================

def section(title):
    print()
    print("=" * 100)
    print(title)
    print("=" * 100)
    sys.stdout.flush()


# ============================================================
# START SPARK
#
# Do not specify .master(...) here.
#
# spark-submit controls the master. This prevents the program
# from silently replacing local[1] with local[2].
# ============================================================

spark = (
    SparkSession.builder
    .appName(
        "GHSL-Population-1975-Sedona-Safe"
    )
    .config(
        "spark.sql.shuffle.partitions",
        "1",
    )
    .config(
        "spark.default.parallelism",
        "1",
    )
    .config(
        "spark.sql.adaptive.enabled",
        "false",
    )
    .config(
        "spark.sql.codegen.wholeStage",
        "false",
    )
    .config(
        "spark.ui.enabled",
        "false",
    )
    .config(
        "spark.sql.files.maxPartitionBytes",
        str(64 * 1024 * 1024),
    )
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")


try:
    # ========================================================
    # REGISTER SEDONA FUNCTIONS
    # ========================================================

    from sedona.spark import SedonaContext

    SedonaContext.create(spark)

    section("SPARK AND SEDONA ENVIRONMENT")

    print("Spark version:", spark.version)
    print("Spark master:", spark.sparkContext.master)
    print(
        "Spark default parallelism:",
        spark.sparkContext.defaultParallelism,
    )
    print(
        "Sedona tile size:",
        f"{TILE_SIZE} x {TILE_SIZE}",
    )
    print(
        "HDFS input:",
        HDFS_INPUT,
    )

    if spark.sparkContext.master != "local[1]":
        raise RuntimeError(
            "Safe job must run with local[1], but Spark "
            f"started with {spark.sparkContext.master}."
        )

    # ========================================================
    # READ HDFS INVENTORY
    # ========================================================

    section("READING HDFS GEOTIFF INVENTORY")

    inventory_df = (
        spark.read
        .format("binaryFile")
        .option(
            "pathGlobFilter",
            "*.tif",
        )
        .load(HDFS_INPUT)
        .select(
            "path",
            "length",
        )
        .orderBy(
            "path",
        )
    )

    inventory_rows = inventory_df.collect()

    shard_count = len(inventory_rows)

    source_bytes = sum(
        int(row["length"])
        for row in inventory_rows
    )

    print("HDFS GeoTIFF shards:", shard_count)
    print("HDFS source bytes:", source_bytes)
    print(
        "HDFS source MiB:",
        source_bytes / (1024.0 * 1024.0),
    )

    if shard_count != EXPECTED_SHARDS:
        raise RuntimeError(
            f"Found {shard_count} GeoTIFF shards; "
            f"expected {EXPECTED_SHARDS}."
        )

    # ========================================================
    # RESULT SCHEMA
    # ========================================================

    shard_result_schema = StructType(
        [
            StructField(
                "epoch",
                IntegerType(),
                False,
            ),
            StructField(
                "filename",
                StringType(),
                False,
            ),
            StructField(
                "hdfs_path",
                StringType(),
                False,
            ),
            StructField(
                "source_bytes",
                LongType(),
                False,
            ),
            StructField(
                "source_width",
                IntegerType(),
                False,
            ),
            StructField(
                "source_height",
                IntegerType(),
                False,
            ),
            StructField(
                "band_count",
                IntegerType(),
                False,
            ),
            StructField(
                "tile_count",
                LongType(),
                False,
            ),
            StructField(
                "total_pixels",
                LongType(),
                False,
            ),
            StructField(
                "valid_pixels",
                LongType(),
                False,
            ),
            StructField(
                "excluded_pixels",
                LongType(),
                False,
            ),
            StructField(
                "zero_pixels",
                LongType(),
                False,
            ),
            StructField(
                "positive_pixels",
                LongType(),
                False,
            ),
            StructField(
                "population_sum",
                DoubleType(),
                False,
            ),
            StructField(
                "sum_squares",
                DoubleType(),
                False,
            ),
            StructField(
                "population_mean",
                DoubleType(),
                False,
            ),
            StructField(
                "population_stddev",
                DoubleType(),
                False,
            ),
            StructField(
                "population_min",
                DoubleType(),
                False,
            ),
            StructField(
                "population_max",
                DoubleType(),
                False,
            ),
            StructField(
                "processing_seconds",
                DoubleType(),
                False,
            ),
        ]
    )

    shard_results = []

    # ========================================================
    # PROCESS ONE HDFS SHARD AT A TIME
    # ========================================================

    section("SEDONA SEQUENTIAL-SHARD DISTRIBUTED-TILE PROCESSING")

    overall_start = time.perf_counter()

    for shard_index, inventory_row in enumerate(
        inventory_rows,
        start=1,
    ):
        shard_path = inventory_row["path"]
        shard_bytes = int(inventory_row["length"])
        filename = shard_path.rsplit("/", 1)[-1]

        print()
        print(
            f"[{shard_index:02d}/{shard_count:02d}] "
            f"{filename}"
        )
        print("    HDFS path:", shard_path)
        print("    Source bytes:", shard_bytes)
        sys.stdout.flush()

        shard_start = time.perf_counter()

        # ----------------------------------------------------
        # Read exactly one GeoTIFF from HDFS
        # ----------------------------------------------------

        one_file_df = (
            spark.read
            .format("binaryFile")
            .load(shard_path)
            .select(
                "path",
                "length",
                "content",
            )
        )

        one_file_df.createOrReplaceTempView(
            "one_binary_geotiff"
        )

        # ----------------------------------------------------
        # Decode one raster and immediately tile it
        #
        # No repartition is performed after raster decoding.
        # This avoids serializing raster objects through a
        # Spark shuffle.
        # ----------------------------------------------------

        tiles_df = spark.sql(
            f"""
            WITH decoded AS
            (
                SELECT
                    path,
                    length AS source_bytes,
                    RS_FromGeoTiff(content) AS raster
                FROM one_binary_geotiff
            )
            SELECT
                path,
                source_bytes,
                RS_Width(raster) AS source_width,
                RS_Height(raster) AS source_height,
                RS_NumBands(raster) AS band_count,
                tile_record.x AS tile_x,
                tile_record.y AS tile_y,
                tile_record.tile AS tile
            FROM decoded
            LATERAL VIEW
                RS_TileExplode(
                    raster,
                    1,
                    {TILE_SIZE},
                    {TILE_SIZE},
                    false
                ) exploded AS tile_record
            """
        )

        tiles_df.createOrReplaceTempView(
            "current_raster_tiles"
        )

        # ----------------------------------------------------
        # Calculate per-tile scalar statistics
        #
        # valid_tile declares -200 as NoData.
        #
        # valid_count:
        # pixels excluding native NoData and -200
        #
        # zero_count:
        # valid_count minus the count obtained when zero is
        # temporarily treated as NoData.
        #
        # This avoids RS_BandAsArray and RS_CountValue.
        # ----------------------------------------------------

        tile_stats_df = spark.sql(
            f"""
            WITH prepared AS
            (
                SELECT
                    path,
                    source_bytes,
                    source_width,
                    source_height,
                    band_count,
                    tile_x,
                    tile_y,
                    tile,
                    RS_SetBandNoDataValue(
                        tile,
                        1,
                        CAST(
                            {INVALID_SENTINEL}
                            AS DOUBLE
                        )
                    ) AS valid_tile
                FROM current_raster_tiles
            ),
            statistics AS
            (
                SELECT
                    path,
                    source_bytes,
                    source_width,
                    source_height,
                    band_count,
                    tile_x,
                    tile_y,

                    CAST(
                        RS_SummaryStats(
                            valid_tile,
                            'count',
                            1,
                            false
                        )
                        AS BIGINT
                    ) AS total_count,

                    CAST(
                        RS_SummaryStats(
                            valid_tile,
                            'count',
                            1,
                            true
                        )
                        AS BIGINT
                    ) AS valid_count,

                    CAST(
                        RS_SummaryStats(
                            valid_tile,
                            'count',
                            1,
                            true
                        )
                        -
                        RS_SummaryStats(
                            RS_SetBandNoDataValue(
                                valid_tile,
                                1,
                                CAST(0.0 AS DOUBLE)
                            ),
                            'count',
                            1,
                            true
                        )
                        AS BIGINT
                    ) AS zero_count,

                    RS_SummaryStats(
                        valid_tile,
                        'sum',
                        1,
                        true
                    ) AS value_sum,

                    RS_SummaryStats(
                        valid_tile,
                        'mean',
                        1,
                        true
                    ) AS value_mean,

                    RS_SummaryStats(
                        valid_tile,
                        'stddev',
                        1,
                        true
                    ) AS value_stddev,

                    RS_SummaryStats(
                        valid_tile,
                        'min',
                        1,
                        true
                    ) AS value_min,

                    RS_SummaryStats(
                        valid_tile,
                        'max',
                        1,
                        true
                    ) AS value_max

                FROM prepared
            )
            SELECT
                path,
                source_bytes,
                source_width,
                source_height,
                band_count,
                tile_x,
                tile_y,
                total_count,
                valid_count,
                total_count - valid_count
                    AS excluded_count,
                zero_count,
                valid_count - zero_count
                    AS positive_count,
                value_sum,
                CASE
                    WHEN valid_count > 0
                    THEN
                        (
                            POWER(value_stddev, 2)
                            + POWER(value_mean, 2)
                        )
                        * valid_count
                    ELSE 0.0
                END AS value_sum_squares,
                value_min,
                value_max
            FROM statistics
            """
        )

        tile_stats_df.createOrReplaceTempView(
            "current_tile_statistics"
        )

        # ----------------------------------------------------
        # Aggregate all tile statistics for this source shard
        # ----------------------------------------------------

        shard_stats_df = spark.sql(
            """
            SELECT
                path,
                MAX(source_bytes) AS source_bytes,
                MAX(source_width) AS source_width,
                MAX(source_height) AS source_height,
                MAX(band_count) AS band_count,
                COUNT(*) AS tile_count,
                SUM(total_count) AS total_pixels,
                SUM(valid_count) AS valid_pixels,
                SUM(excluded_count) AS excluded_pixels,
                SUM(zero_count) AS zero_pixels,
                SUM(positive_count) AS positive_pixels,
                SUM(value_sum) AS population_sum,
                SUM(value_sum_squares) AS sum_squares,
                MIN(
                    CASE
                        WHEN valid_count > 0
                        THEN value_min
                        ELSE NULL
                    END
                ) AS population_min,
                MAX(
                    CASE
                        WHEN valid_count > 0
                        THEN value_max
                        ELSE NULL
                    END
                ) AS population_max
            FROM current_tile_statistics
            GROUP BY path
            """
        )

        # Only one scalar row crosses into Python.
        shard_row = shard_stats_df.first()

        if shard_row is None:
            raise RuntimeError(
                f"No statistics were produced for {filename}."
            )

        valid_pixels = int(
            shard_row["valid_pixels"]
        )

        population_sum = float(
            shard_row["population_sum"]
            or 0.0
        )

        sum_squares = float(
            shard_row["sum_squares"]
            or 0.0
        )

        if valid_pixels > 0:
            population_mean = (
                population_sum / valid_pixels
            )

            population_variance = max(
                0.0,
                sum_squares / valid_pixels
                - population_mean * population_mean,
            )

            population_stddev = math.sqrt(
                population_variance
            )

            population_min = float(
                shard_row["population_min"]
            )

            population_max = float(
                shard_row["population_max"]
            )
        else:
            population_mean = 0.0
            population_stddev = 0.0
            population_min = 0.0
            population_max = 0.0

        processing_seconds = (
            time.perf_counter()
            - shard_start
        )

        result_tuple = (
            EPOCH,
            filename,
            shard_path,
            int(shard_row["source_bytes"]),
            int(shard_row["source_width"]),
            int(shard_row["source_height"]),
            int(shard_row["band_count"]),
            int(shard_row["tile_count"]),
            int(shard_row["total_pixels"]),
            valid_pixels,
            int(shard_row["excluded_pixels"]),
            int(shard_row["zero_pixels"]),
            int(shard_row["positive_pixels"]),
            population_sum,
            sum_squares,
            population_mean,
            population_stddev,
            population_min,
            population_max,
            float(processing_seconds),
        )

        shard_results.append(
            result_tuple
        )

        print(
            "    Raster dimensions:",
            f"{result_tuple[4]} x {result_tuple[5]}",
        )
        print(
            "    Tiles:",
            result_tuple[7],
        )
        print(
            "    Total pixels:",
            result_tuple[8],
        )
        print(
            "    Valid pixels:",
            result_tuple[9],
        )
        print(
            "    Excluded pixels:",
            result_tuple[10],
        )
        print(
            "    Zero pixels:",
            result_tuple[11],
        )
        print(
            "    Positive pixels:",
            result_tuple[12],
        )
        print(
            "    Population sum:",
            result_tuple[13],
        )
        print(
            "    Processing seconds:",
            round(
                processing_seconds,
                3,
            ),
        )
        sys.stdout.flush()

        # Remove temporary views before processing the next
        # source raster.
        for view_name in [
            "one_binary_geotiff",
            "current_raster_tiles",
            "current_tile_statistics",
        ]:
            try:
                spark.catalog.dropTempView(
                    view_name
                )
            except Exception:
                pass

        # Request JVM garbage collection only between complete
        # source shards, never during a task.
        try:
            spark.sparkContext._jvm.System.gc()
        except Exception:
            pass

    overall_seconds = (
        time.perf_counter()
        - overall_start
    )

    # ========================================================
    # CREATE SCALAR SPARK DATAFRAME
    # ========================================================

    section("CREATING SHARD STATISTICS DATAFRAME")

    shard_results_df = spark.createDataFrame(
        shard_results,
        schema=shard_result_schema,
    )

    shard_results_df.orderBy(
        "filename"
    ).show(
        n=EXPECTED_SHARDS,
        truncate=False,
    )

    # ========================================================
    # WRITE SHARD STATISTICS TO HDFS PARQUET
    # ========================================================

    section("WRITING SHARD STATISTICS TO HDFS")

    (
        shard_results_df
        .coalesce(1)
        .write
        .mode("overwrite")
        .parquet(HDFS_SHARD_OUTPUT)
    )

    print(
        "Shard statistics output:",
        HDFS_SHARD_OUTPUT,
    )

    # ========================================================
    # AGGREGATE COMPLETE EPOCH USING SPARK
    # ========================================================

    section("AGGREGATING COMPLETE 1975 EPOCH")

    epoch_raw_df = (
        shard_results_df
        .agg(
            F.count("*").alias(
                "shard_count"
            ),
            F.sum("source_bytes").alias(
                "source_bytes"
            ),
            F.sum("tile_count").alias(
                "tile_count"
            ),
            F.sum("total_pixels").alias(
                "total_pixels"
            ),
            F.sum("valid_pixels").alias(
                "valid_pixels"
            ),
            F.sum("excluded_pixels").alias(
                "excluded_pixels"
            ),
            F.sum("zero_pixels").alias(
                "zero_pixels"
            ),
            F.sum("positive_pixels").alias(
                "positive_pixels"
            ),
            F.sum("population_sum").alias(
                "population_sum"
            ),
            F.sum("sum_squares").alias(
                "sum_squares"
            ),
            F.min("population_min").alias(
                "population_min"
            ),
            F.max("population_max").alias(
                "population_max"
            ),
            F.sum("processing_seconds").alias(
                "summed_shard_processing_seconds"
            ),
        )
        .withColumn(
            "epoch",
            F.lit(EPOCH),
        )
        .withColumn(
            "product",
            F.lit("population"),
        )
        .withColumn(
            "resolution_m",
            F.lit(100),
        )
        .withColumn(
            "tile_size",
            F.lit(TILE_SIZE),
        )
        .withColumn(
            "processing_architecture",
            F.lit(
                "HDFS -> Spark binaryFile -> "
                "Sedona RS_FromGeoTiff -> "
                "Sedona RS_TileExplode -> "
                "Sedona RS_SummaryStats -> "
                "Spark aggregation -> HDFS Parquet"
            ),
        )
        .withColumn(
            "population_mean",
            F.col("population_sum")
            / F.col("valid_pixels"),
        )
        .withColumn(
            "population_variance",
            F.greatest(
                F.lit(0.0),
                F.col("sum_squares")
                / F.col("valid_pixels")
                - F.pow(
                    F.col("population_sum")
                    / F.col("valid_pixels"),
                    2,
                ),
            ),
        )
        .withColumn(
            "population_stddev",
            F.sqrt(
                F.col("population_variance")
            ),
        )
        .withColumn(
            "wall_clock_seconds",
            F.lit(float(overall_seconds)),
        )
    )

    epoch_columns = [
        "epoch",
        "product",
        "resolution_m",
        "tile_size",
        "shard_count",
        "source_bytes",
        "tile_count",
        "total_pixels",
        "valid_pixels",
        "excluded_pixels",
        "zero_pixels",
        "positive_pixels",
        "population_sum",
        "population_mean",
        "population_stddev",
        "population_min",
        "population_max",
        "summed_shard_processing_seconds",
        "wall_clock_seconds",
        "processing_architecture",
    ]

    epoch_df = epoch_raw_df.select(
        *epoch_columns
    )

    epoch_df.show(
        truncate=False
    )

    # ========================================================
    # WRITE EPOCH RESULT TO HDFS PARQUET
    # ========================================================

    section("WRITING EPOCH STATISTICS TO HDFS")

    (
        epoch_df
        .coalesce(1)
        .write
        .mode("overwrite")
        .parquet(HDFS_EPOCH_OUTPUT)
    )

    print(
        "Epoch statistics output:",
        HDFS_EPOCH_OUTPUT,
    )

    # ========================================================
    # BASIC INTERNAL VALIDATION
    # ========================================================

    section("INTERNAL VALIDATION")

    epoch_row = epoch_df.first()

    checks = {
        "shard_count": (
            int(epoch_row["shard_count"])
            == EXPECTED_SHARDS
        ),
        "source_bytes_positive": (
            int(epoch_row["source_bytes"]) > 0
        ),
        "total_pixel_identity": (
            int(epoch_row["total_pixels"])
            ==
            int(epoch_row["valid_pixels"])
            + int(epoch_row["excluded_pixels"])
        ),
        "valid_pixel_identity": (
            int(epoch_row["valid_pixels"])
            ==
            int(epoch_row["zero_pixels"])
            + int(epoch_row["positive_pixels"])
        ),
        "population_sum_positive": (
            float(epoch_row["population_sum"]) > 0.0
        ),
        "finite_mean": math.isfinite(
            float(epoch_row["population_mean"])
        ),
        "finite_stddev": math.isfinite(
            float(epoch_row["population_stddev"])
        ),
    }

    failed_checks = [
        name
        for name, passed in checks.items()
        if not passed
    ]

    for name, passed in checks.items():
        print(
            f"{name}:",
            "PASS" if passed else "FAIL",
        )

    if failed_checks:
        raise RuntimeError(
            "Internal validation failed: "
            + ", ".join(failed_checks)
        )

    section("JOB COMPLETED SUCCESSFULLY")

    print(
        "The 1975 population epoch was processed "
        "with Spark, Sedona and HDFS."
    )
    print(
        "Total wall-clock seconds:",
        round(
            overall_seconds,
            3,
        ),
    )

except Exception:
    print()
    print("=" * 100)
    print("JOB FAILED")
    print("=" * 100)
    traceback.print_exc()
    sys.stdout.flush()
    sys.stderr.flush()
    raise

finally:
    try:
        spark.stop()
    except Exception:
        pass
'''


# ============================================================
# WRITE CLEAN PROGRAM LOCALLY
# ============================================================

REMOTE_PROGRAM = textwrap.dedent(
    REMOTE_PROGRAM
).lstrip()

# Catch Python syntax errors before uploading.
compile(
    REMOTE_PROGRAM,
    str(LOCAL_SCRIPT),
    "exec",
)

LOCAL_SCRIPT.write_text(
    REMOTE_PROGRAM,
    encoding="utf-8",
)

print(
    "Local program created:",
    LOCAL_SCRIPT,
)

print(
    "Program size:",
    LOCAL_SCRIPT.stat().st_size,
    "bytes",
)

print(
    "Local syntax check:",
    "PASSED",
)


# ============================================================
# STATIC ARCHITECTURAL SAFETY CHECKS
# ============================================================

# ============================================================
# STATIC ARCHITECTURAL SAFETY CHECKS
#
# Use Python tokenization so comments and string literals do
# not create false positives.
# ============================================================

import ast
import io
import re
import tokenize


# Confirm that the generated program parses as valid Python.
parsed_program = ast.parse(
    REMOTE_PROGRAM,
    filename=str(LOCAL_SCRIPT),
)


# ============================================================
# Extract executable Python tokens only
#
# Comments are removed before checking forbidden expressions.
# This prevents comments such as:
#
#     # no RS_BandAsArray
#
# from being mistaken for executable function calls.
# ============================================================

executable_tokens = []

token_stream = tokenize.generate_tokens(
    io.StringIO(
        REMOTE_PROGRAM
    ).readline
)

for token_info in token_stream:
    if token_info.type in {
        tokenize.COMMENT,
        tokenize.ENCODING,
    }:
        continue

    executable_tokens.append(
        token_info.string
    )

program_without_comments = tokenize.untokenize(
    [
        token_info
        for token_info in tokenize.generate_tokens(
            io.StringIO(
                REMOTE_PROGRAM
            ).readline
        )
        if token_info.type != tokenize.COMMENT
    ]
)


# ============================================================
# Check required architecture fragments
#
# These operations mostly occur inside Spark SQL strings, so
# checking the complete program text is correct here.
# ============================================================

required_fragments = [
    "RS_FromGeoTiff",
    "RS_TileExplode",
    "RS_SetBandNoDataValue",
    "RS_SummaryStats",
    'format("binaryFile")',
    ".parquet(",
]

missing_fragments = [
    fragment
    for fragment in required_fragments
    if fragment not in REMOTE_PROGRAM
]

if missing_fragments:
    raise RuntimeError(
        "Required program fragments are missing: "
        + ", ".join(missing_fragments)
    )


# ============================================================
# Check forbidden Sedona SQL function calls
#
# Match actual function syntax only:
#
#     RS_BandAsArray(
#
# A comment containing only the function name does not match.
# ============================================================

forbidden_sql_call_patterns = {
    "RS_BandAsArray call": (
        r"\bRS_BandAsArray\s*\("
    ),
    "RS_CountValue call": (
        r"\bRS_CountValue\s*\("
    ),
}

for operation, pattern in forbidden_sql_call_patterns.items():
    if re.search(
        pattern,
        program_without_comments,
    ):
        raise RuntimeError(
            f"Forbidden SQL operation found: {operation}"
        )


# ============================================================
# Inspect actual Python calls using the AST
# ============================================================

called_names = set()
called_attributes = set()

for node in ast.walk(parsed_program):
    if not isinstance(node, ast.Call):
        continue

    if isinstance(node.func, ast.Name):
        called_names.add(
            node.func.id
        )

    elif isinstance(node.func, ast.Attribute):
        called_attributes.add(
            node.func.attr
        )


forbidden_python_names = {
    "finite_statistics_udf",
}

forbidden_python_attributes = {
    "as_numpy",
    "cache",
    "persist",
    "localCheckpoint",
    "repartition",
}

found_forbidden_names = sorted(
    forbidden_python_names
    & called_names
)

found_forbidden_attributes = sorted(
    forbidden_python_attributes
    & called_attributes
)

if found_forbidden_names:
    raise RuntimeError(
        "Forbidden Python calls found: "
        + ", ".join(found_forbidden_names)
    )

if found_forbidden_attributes:
    raise RuntimeError(
        "Forbidden DataFrame/raster calls found: "
        + ", ".join(found_forbidden_attributes)
    )


# ============================================================
# Confirm that the program does not override Spark master
# ============================================================

master_override_pattern = (
    r"\.master\s*\("
)

if re.search(
    master_override_pattern,
    program_without_comments,
):
    raise RuntimeError(
        "The generated program overrides the Spark master. "
        "Remove the .master(...) call."
    )


# ============================================================
# Confirm that external raster engines are not imported
# ============================================================

imported_modules = set()

for node in ast.walk(parsed_program):
    if isinstance(node, ast.Import):
        for alias in node.names:
            imported_modules.add(
                alias.name.split(".")[0]
            )

    elif isinstance(node, ast.ImportFrom):
        if node.module:
            imported_modules.add(
                node.module.split(".")[0]
            )

forbidden_imports = {
    "numpy",
    "rasterio",
}

found_forbidden_imports = sorted(
    forbidden_imports
    & imported_modules
)

if found_forbidden_imports:
    raise RuntimeError(
        "Forbidden imports found: "
        + ", ".join(found_forbidden_imports)
    )


# ============================================================
# Final validation report
# ============================================================

print(
    "Architecture safety check:",
    "PASSED",
)

print(
    "Required architecture:",
    "HDFS -> Spark binaryFile -> Sedona raster tiles "
    "-> Spark aggregation -> HDFS Parquet",
)

print(
    "RS_BandAsArray runtime call:",
    "ABSENT",
)

print(
    "RS_CountValue runtime call:",
    "ABSENT",
)

print(
    "Python raster UDF:",
    "ABSENT",
)

print(
    "Raster-to-NumPy conversion:",
    "ABSENT",
)

print(
    "Raster shuffle/cache/checkpoint:",
    "ABSENT",
)

print(
    "Internal Spark master override:",
    "ABSENT",
)

Local program created: /content/ghsl_population_1975_sedona_safe.py
Program size: 26206 bytes
Local syntax check: PASSED
Architecture safety check: PASSED
Required architecture: HDFS -> Spark binaryFile -> Sedona raster tiles -> Spark aggregation -> HDFS Parquet
RS_BandAsArray runtime call: ABSENT
RS_CountValue runtime call: ABSENT
Python raster UDF: ABSENT
Raster-to-NumPy conversion: ABSENT
Raster shuffle/cache/checkpoint: ABSENT
Internal Spark master override: ABSENT


In [ ]:
from pathlib import Path
import subprocess
import textwrap


# ============================================================
# NOTEBOOK CONFIGURATION
# ============================================================

LOCAL_SCRIPT = Path(
    "/content/ghsl_population_1975_sedona_safe.py"
)

REMOTE_SCRIPT = (
    "/home/ubuntu/ghsl_population_1975_sedona_safe.py"
)

REMOTE_EXEC = "/content/drive/MyDrive/remote_exec.sh"
SSH_KEY = "/content/hadoop-hdfs.pem"

AWS_USER = "ubuntu"
AWS_HOST = "54.92.192.154"


# ============================================================
# CLEAN STANDALONE SPARK + SEDONA + HDFS PROGRAM
#
# Important design:
#
# 1. Spark reads the GeoTIFF inventory directly from HDFS.
# 2. Only one source GeoTIFF is processed at a time.
# 3. Sedona decodes the selected GeoTIFF in the JVM.
# 4. Sedona splits it into 256 x 256 raster tiles.
# 5. Sedona calculates scalar statistics per tile.
# 6. Spark aggregates the tile statistics.
# 7. Only one tiny scalar result row is collected to Python.
# 8. Final results are written to HDFS as Parquet.
#
# There is:
#
# - no Rasterio
# - no NumPy
# - no Python UDF
# - no RS_BandAsArray
# - no RS_CountValue
# - no raster cache
# - no raster checkpoint
# - no raster repartition/shuffle
# ============================================================

REMOTE_PROGRAM = r'''
import math
import sys
import time
import traceback

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    LongType,
    IntegerType,
    DoubleType,
)


# ============================================================
# CONSTANTS
# ============================================================

EPOCH = 1975
EXPECTED_SHARDS = 15

TILE_SIZE = 256
INVALID_SENTINEL = -200.0

HDFS_INPUT = (
    "hdfs://localhost:9000"
    "/data/ghsl/raw/india/population/epoch=1975"
)

HDFS_OUTPUT_BASE = (
    "hdfs://localhost:9000"
    "/data/ghsl/processed/india/population/epoch=1975"
)

HDFS_SHARD_OUTPUT = (
    HDFS_OUTPUT_BASE
    + "/sedona_shard_statistics"
)

HDFS_EPOCH_OUTPUT = (
    HDFS_OUTPUT_BASE
    + "/sedona_epoch_statistics"
)


# ============================================================
# FORMATTING HELPER
# ============================================================

def section(title):
    print()
    print("=" * 100)
    print(title)
    print("=" * 100)
    sys.stdout.flush()


# ============================================================
# START SPARK
#
# Do not specify .master(...) here.
#
# spark-submit controls the master. This prevents the program
# from silently replacing local[1] with local[2].
# ============================================================

spark = (
    SparkSession.builder
    .appName(
        "GHSL-Population-1975-Sedona-Safe"
    )
    .config(
        "spark.sql.shuffle.partitions",
        "1",
    )
    .config(
        "spark.default.parallelism",
        "1",
    )
    .config(
        "spark.sql.adaptive.enabled",
        "false",
    )
    .config(
        "spark.sql.codegen.wholeStage",
        "false",
    )
    .config(
        "spark.ui.enabled",
        "false",
    )
    .config(
        "spark.sql.files.maxPartitionBytes",
        str(64 * 1024 * 1024),
    )
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")


try:
    # ========================================================
    # REGISTER SEDONA FUNCTIONS
    # ========================================================

    from sedona.spark import SedonaContext

    SedonaContext.create(spark)

    section("SPARK AND SEDONA ENVIRONMENT")

    print("Spark version:", spark.version)
    print("Spark master:", spark.sparkContext.master)
    print(
        "Spark default parallelism:",
        spark.sparkContext.defaultParallelism,
    )
    print(
        "Sedona tile size:",
        f"{TILE_SIZE} x {TILE_SIZE}",
    )
    print(
        "HDFS input:",
        HDFS_INPUT,
    )

    if spark.sparkContext.master != "local[1]":
        raise RuntimeError(
            "Safe job must run with local[1], but Spark "
            f"started with {spark.sparkContext.master}."
        )

    # ========================================================
    # READ HDFS INVENTORY
    # ========================================================

    section("READING HDFS GEOTIFF INVENTORY")

    inventory_df = (
        spark.read
        .format("binaryFile")
        .option(
            "pathGlobFilter",
            "*.tif",
        )
        .load(HDFS_INPUT)
        .select(
            "path",
            "length",
        )
        .orderBy(
            "path",
        )
    )

    inventory_rows = inventory_df.collect()

    shard_count = len(inventory_rows)

    source_bytes = sum(
        int(row["length"])
        for row in inventory_rows
    )

    print("HDFS GeoTIFF shards:", shard_count)
    print("HDFS source bytes:", source_bytes)
    print(
        "HDFS source MiB:",
        source_bytes / (1024.0 * 1024.0),
    )

    if shard_count != EXPECTED_SHARDS:
        raise RuntimeError(
            f"Found {shard_count} GeoTIFF shards; "
            f"expected {EXPECTED_SHARDS}."
        )

    # ========================================================
    # RESULT SCHEMA
    # ========================================================

    shard_result_schema = StructType(
        [
            StructField(
                "epoch",
                IntegerType(),
                False,
            ),
            StructField(
                "filename",
                StringType(),
                False,
            ),
            StructField(
                "hdfs_path",
                StringType(),
                False,
            ),
            StructField(
                "source_bytes",
                LongType(),
                False,
            ),
            StructField(
                "source_width",
                IntegerType(),
                False,
            ),
            StructField(
                "source_height",
                IntegerType(),
                False,
            ),
            StructField(
                "band_count",
                IntegerType(),
                False,
            ),
            StructField(
                "tile_count",
                LongType(),
                False,
            ),
            StructField(
                "total_pixels",
                LongType(),
                False,
            ),
            StructField(
                "valid_pixels",
                LongType(),
                False,
            ),
            StructField(
                "excluded_pixels",
                LongType(),
                False,
            ),
            StructField(
                "zero_pixels",
                LongType(),
                False,
            ),
            StructField(
                "positive_pixels",
                LongType(),
                False,
            ),
            StructField(
                "population_sum",
                DoubleType(),
                False,
            ),
            StructField(
                "sum_squares",
                DoubleType(),
                False,
            ),
            StructField(
                "population_mean",
                DoubleType(),
                False,
            ),
            StructField(
                "population_stddev",
                DoubleType(),
                False,
            ),
            StructField(
                "population_min",
                DoubleType(),
                False,
            ),
            StructField(
                "population_max",
                DoubleType(),
                False,
            ),
            StructField(
                "processing_seconds",
                DoubleType(),
                False,
            ),
        ]
    )

    shard_results = []

    # ========================================================
    # PROCESS ONE HDFS SHARD AT A TIME
    # ========================================================

    section("SEDONA SEQUENTIAL-SHARD DISTRIBUTED-TILE PROCESSING")

    overall_start = time.perf_counter()

    for shard_index, inventory_row in enumerate(
        inventory_rows,
        start=1,
    ):
        shard_path = inventory_row["path"]
        shard_bytes = int(inventory_row["length"])
        filename = shard_path.rsplit("/", 1)[-1]

        print()
        print(
            f"[{shard_index:02d}/{shard_count:02d}] "
            f"{filename}"
        )
        print("    HDFS path:", shard_path)
        print("    Source bytes:", shard_bytes)
        sys.stdout.flush()

        shard_start = time.perf_counter()

        # ----------------------------------------------------
        # Read exactly one GeoTIFF from HDFS
        # ----------------------------------------------------

        one_file_df = (
            spark.read
            .format("binaryFile")
            .load(shard_path)
            .select(
                "path",
                "length",
                "content",
            )
        )

        one_file_df.createOrReplaceTempView(
            "one_binary_geotiff"
        )

        # ----------------------------------------------------
        # Decode one raster and immediately tile it
        #
        # No repartition is performed after raster decoding.
        # This avoids serializing raster objects through a
        # Spark shuffle.
        # ----------------------------------------------------

        tiles_df = spark.sql(
            f"""
            WITH decoded AS
            (
                SELECT
                    path,
                    length AS source_bytes,
                    RS_FromGeoTiff(content) AS raster
                FROM one_binary_geotiff
            )
            SELECT
                path,
                source_bytes,
                RS_Width(raster) AS source_width,
                RS_Height(raster) AS source_height,
                RS_NumBands(raster) AS band_count,
                tile_record.x AS tile_x,
                tile_record.y AS tile_y,
                tile_record.tile AS tile
            FROM decoded
            LATERAL VIEW
                RS_TileExplode(
                    raster,
                    1,
                    {TILE_SIZE},
                    {TILE_SIZE},
                    false
                ) exploded AS tile_record
            """
        )

        tiles_df.createOrReplaceTempView(
            "current_raster_tiles"
        )

        # ----------------------------------------------------
        # Calculate per-tile scalar statistics
        #
        # valid_tile declares -200 as NoData.
        #
        # valid_count:
        # pixels excluding native NoData and -200
        #
        # zero_count:
        # valid_count minus the count obtained when zero is
        # temporarily treated as NoData.
        #
        # This avoids RS_BandAsArray and RS_CountValue.
        # ----------------------------------------------------

        tile_stats_df = spark.sql(
            f"""
            WITH prepared AS
            (
                SELECT
                    path,
                    source_bytes,
                    source_width,
                    source_height,
                    band_count,
                    tile_x,
                    tile_y,
                    tile,
                    RS_SetBandNoDataValue(
                        tile,
                        1,
                        CAST(
                            {INVALID_SENTINEL}
                            AS DOUBLE
                        )
                    ) AS valid_tile
                FROM current_raster_tiles
            ),
            statistics AS
            (
                SELECT
                    path,
                    source_bytes,
                    source_width,
                    source_height,
                    band_count,
                    tile_x,
                    tile_y,

                    CAST(
                        RS_SummaryStats(
                            valid_tile,
                            'count',
                            1,
                            false
                        )
                        AS BIGINT
                    ) AS total_count,

                    CAST(
                        RS_SummaryStats(
                            valid_tile,
                            'count',
                            1,
                            true
                        )
                        AS BIGINT
                    ) AS valid_count,

                    CAST(
                        RS_SummaryStats(
                            valid_tile,
                            'count',
                            1,
                            true
                        )
                        -
                        RS_SummaryStats(
                            RS_SetBandNoDataValue(
                                valid_tile,
                                1,
                                CAST(0.0 AS DOUBLE)
                            ),
                            'count',
                            1,
                            true
                        )
                        AS BIGINT
                    ) AS zero_count,

                    RS_SummaryStats(
                        valid_tile,
                        'sum',
                        1,
                        true
                    ) AS value_sum,

                    RS_SummaryStats(
                        valid_tile,
                        'mean',
                        1,
                        true
                    ) AS value_mean,

                    RS_SummaryStats(
                        valid_tile,
                        'stddev',
                        1,
                        true
                    ) AS value_stddev,

                    RS_SummaryStats(
                        valid_tile,
                        'min',
                        1,
                        true
                    ) AS value_min,

                    RS_SummaryStats(
                        valid_tile,
                        'max',
                        1,
                        true
                    ) AS value_max

                FROM prepared
            )
            SELECT
                path,
                source_bytes,
                source_width,
                source_height,
                band_count,
                tile_x,
                tile_y,
                total_count,
                valid_count,
                total_count - valid_count
                    AS excluded_count,
                zero_count,
                valid_count - zero_count
                    AS positive_count,
                value_sum,
                CASE
                    WHEN valid_count > 0
                    THEN
                        (
                            POWER(value_stddev, 2)
                            + POWER(value_mean, 2)
                        )
                        * valid_count
                    ELSE 0.0
                END AS value_sum_squares,
                value_min,
                value_max
            FROM statistics
            """
        )

        tile_stats_df.createOrReplaceTempView(
            "current_tile_statistics"
        )

        # ----------------------------------------------------
        # Aggregate all tile statistics for this source shard
        # ----------------------------------------------------

        shard_stats_df = spark.sql(
            """
            SELECT
                path,
                MAX(source_bytes) AS source_bytes,
                MAX(source_width) AS source_width,
                MAX(source_height) AS source_height,
                MAX(band_count) AS band_count,
                COUNT(*) AS tile_count,
                SUM(total_count) AS total_pixels,
                SUM(valid_count) AS valid_pixels,
                SUM(excluded_count) AS excluded_pixels,
                SUM(zero_count) AS zero_pixels,
                SUM(positive_count) AS positive_pixels,
                SUM(value_sum) AS population_sum,
                SUM(value_sum_squares) AS sum_squares,
                MIN(
                    CASE
                        WHEN valid_count > 0
                        THEN value_min
                        ELSE NULL
                    END
                ) AS population_min,
                MAX(
                    CASE
                        WHEN valid_count > 0
                        THEN value_max
                        ELSE NULL
                    END
                ) AS population_max
            FROM current_tile_statistics
            GROUP BY path
            """
        )

        # Only one scalar row crosses into Python.
        shard_row = shard_stats_df.first()

        if shard_row is None:
            raise RuntimeError(
                f"No statistics were produced for {filename}."
            )

        valid_pixels = int(
            shard_row["valid_pixels"]
        )

        population_sum = float(
            shard_row["population_sum"]
            or 0.0
        )

        sum_squares = float(
            shard_row["sum_squares"]
            or 0.0
        )

        if valid_pixels > 0:
            population_mean = (
                population_sum / valid_pixels
            )

            population_variance = max(
                0.0,
                sum_squares / valid_pixels
                - population_mean * population_mean,
            )

            population_stddev = math.sqrt(
                population_variance
            )

            population_min = float(
                shard_row["population_min"]
            )

            population_max = float(
                shard_row["population_max"]
            )
        else:
            population_mean = 0.0
            population_stddev = 0.0
            population_min = 0.0
            population_max = 0.0

        processing_seconds = (
            time.perf_counter()
            - shard_start
        )

        result_tuple = (
            EPOCH,
            filename,
            shard_path,
            int(shard_row["source_bytes"]),
            int(shard_row["source_width"]),
            int(shard_row["source_height"]),
            int(shard_row["band_count"]),
            int(shard_row["tile_count"]),
            int(shard_row["total_pixels"]),
            valid_pixels,
            int(shard_row["excluded_pixels"]),
            int(shard_row["zero_pixels"]),
            int(shard_row["positive_pixels"]),
            population_sum,
            sum_squares,
            population_mean,
            population_stddev,
            population_min,
            population_max,
            float(processing_seconds),
        )

        shard_results.append(
            result_tuple
        )

        print(
            "    Raster dimensions:",
            f"{result_tuple[4]} x {result_tuple[5]}",
        )
        print(
            "    Tiles:",
            result_tuple[7],
        )
        print(
            "    Total pixels:",
            result_tuple[8],
        )
        print(
            "    Valid pixels:",
            result_tuple[9],
        )
        print(
            "    Excluded pixels:",
            result_tuple[10],
        )
        print(
            "    Zero pixels:",
            result_tuple[11],
        )
        print(
            "    Positive pixels:",
            result_tuple[12],
        )
        print(
            "    Population sum:",
            result_tuple[13],
        )
        print(
            "    Processing seconds:",
            round(
                processing_seconds,
                3,
            ),
        )
        sys.stdout.flush()

        # Remove temporary views before processing the next
        # source raster.
        for view_name in [
            "one_binary_geotiff",
            "current_raster_tiles",
            "current_tile_statistics",
        ]:
            try:
                spark.catalog.dropTempView(
                    view_name
                )
            except Exception:
                pass

        # Request JVM garbage collection only between complete
        # source shards, never during a task.
        try:
            spark.sparkContext._jvm.System.gc()
        except Exception:
            pass

    overall_seconds = (
        time.perf_counter()
        - overall_start
    )

    # ========================================================
    # CREATE SCALAR SPARK DATAFRAME
    # ========================================================

    section("CREATING SHARD STATISTICS DATAFRAME")

    shard_results_df = spark.createDataFrame(
        shard_results,
        schema=shard_result_schema,
    )

    shard_results_df.orderBy(
        "filename"
    ).show(
        n=EXPECTED_SHARDS,
        truncate=False,
    )

    # ========================================================
    # WRITE SHARD STATISTICS TO HDFS PARQUET
    # ========================================================

    section("WRITING SHARD STATISTICS TO HDFS")

    (
        shard_results_df
        .coalesce(1)
        .write
        .mode("overwrite")
        .parquet(HDFS_SHARD_OUTPUT)
    )

    print(
        "Shard statistics output:",
        HDFS_SHARD_OUTPUT,
    )

    # ========================================================
    # AGGREGATE COMPLETE EPOCH USING SPARK
    # ========================================================

    section("AGGREGATING COMPLETE 1975 EPOCH")

    epoch_raw_df = (
        shard_results_df
        .agg(
            F.count("*").alias(
                "shard_count"
            ),
            F.sum("source_bytes").alias(
                "source_bytes"
            ),
            F.sum("tile_count").alias(
                "tile_count"
            ),
            F.sum("total_pixels").alias(
                "total_pixels"
            ),
            F.sum("valid_pixels").alias(
                "valid_pixels"
            ),
            F.sum("excluded_pixels").alias(
                "excluded_pixels"
            ),
            F.sum("zero_pixels").alias(
                "zero_pixels"
            ),
            F.sum("positive_pixels").alias(
                "positive_pixels"
            ),
            F.sum("population_sum").alias(
                "population_sum"
            ),
            F.sum("sum_squares").alias(
                "sum_squares"
            ),
            F.min("population_min").alias(
                "population_min"
            ),
            F.max("population_max").alias(
                "population_max"
            ),
            F.sum("processing_seconds").alias(
                "summed_shard_processing_seconds"
            ),
        )
        .withColumn(
            "epoch",
            F.lit(EPOCH),
        )
        .withColumn(
            "product",
            F.lit("population"),
        )
        .withColumn(
            "resolution_m",
            F.lit(100),
        )
        .withColumn(
            "tile_size",
            F.lit(TILE_SIZE),
        )
        .withColumn(
            "processing_architecture",
            F.lit(
                "HDFS -> Spark binaryFile -> "
                "Sedona RS_FromGeoTiff -> "
                "Sedona RS_TileExplode -> "
                "Sedona RS_SummaryStats -> "
                "Spark aggregation -> HDFS Parquet"
            ),
        )
        .withColumn(
            "population_mean",
            F.col("population_sum")
            / F.col("valid_pixels"),
        )
        .withColumn(
            "population_variance",
            F.greatest(
                F.lit(0.0),
                F.col("sum_squares")
                / F.col("valid_pixels")
                - F.pow(
                    F.col("population_sum")
                    / F.col("valid_pixels"),
                    2,
                ),
            ),
        )
        .withColumn(
            "population_stddev",
            F.sqrt(
                F.col("population_variance")
            ),
        )
        .withColumn(
            "wall_clock_seconds",
            F.lit(float(overall_seconds)),
        )
    )

    epoch_columns = [
        "epoch",
        "product",
        "resolution_m",
        "tile_size",
        "shard_count",
        "source_bytes",
        "tile_count",
        "total_pixels",
        "valid_pixels",
        "excluded_pixels",
        "zero_pixels",
        "positive_pixels",
        "population_sum",
        "population_mean",
        "population_stddev",
        "population_min",
        "population_max",
        "summed_shard_processing_seconds",
        "wall_clock_seconds",
        "processing_architecture",
    ]

    epoch_df = epoch_raw_df.select(
        *epoch_columns
    )

    epoch_df.show(
        truncate=False
    )

    # ========================================================
    # WRITE EPOCH RESULT TO HDFS PARQUET
    # ========================================================

    section("WRITING EPOCH STATISTICS TO HDFS")

    (
        epoch_df
        .coalesce(1)
        .write
        .mode("overwrite")
        .parquet(HDFS_EPOCH_OUTPUT)
    )

    print(
        "Epoch statistics output:",
        HDFS_EPOCH_OUTPUT,
    )

    # ========================================================
    # BASIC INTERNAL VALIDATION
    # ========================================================

    section("INTERNAL VALIDATION")

    epoch_row = epoch_df.first()

    checks = {
        "shard_count": (
            int(epoch_row["shard_count"])
            == EXPECTED_SHARDS
        ),
        "source_bytes_positive": (
            int(epoch_row["source_bytes"]) > 0
        ),
        "total_pixel_identity": (
            int(epoch_row["total_pixels"])
            ==
            int(epoch_row["valid_pixels"])
            + int(epoch_row["excluded_pixels"])
        ),
        "valid_pixel_identity": (
            int(epoch_row["valid_pixels"])
            ==
            int(epoch_row["zero_pixels"])
            + int(epoch_row["positive_pixels"])
        ),
        "population_sum_positive": (
            float(epoch_row["population_sum"]) > 0.0
        ),
        "finite_mean": math.isfinite(
            float(epoch_row["population_mean"])
        ),
        "finite_stddev": math.isfinite(
            float(epoch_row["population_stddev"])
        ),
    }

    failed_checks = [
        name
        for name, passed in checks.items()
        if not passed
    ]

    for name, passed in checks.items():
        print(
            f"{name}:",
            "PASS" if passed else "FAIL",
        )

    if failed_checks:
        raise RuntimeError(
            "Internal validation failed: "
            + ", ".join(failed_checks)
        )

    section("JOB COMPLETED SUCCESSFULLY")

    print(
        "The 1975 population epoch was processed "
        "with Spark, Sedona and HDFS."
    )
    print(
        "Total wall-clock seconds:",
        round(
            overall_seconds,
            3,
        ),
    )

except Exception:
    print()
    print("=" * 100)
    print("JOB FAILED")
    print("=" * 100)
    traceback.print_exc()
    sys.stdout.flush()
    sys.stderr.flush()
    raise

finally:
    try:
        spark.stop()
    except Exception:
        pass
'''


# ============================================================
# WRITE CLEAN PROGRAM LOCALLY
# ============================================================

REMOTE_PROGRAM = textwrap.dedent(
    REMOTE_PROGRAM
).lstrip()

# Catch Python syntax errors before uploading.
compile(
    REMOTE_PROGRAM,
    str(LOCAL_SCRIPT),
    "exec",
)

LOCAL_SCRIPT.write_text(
    REMOTE_PROGRAM,
    encoding="utf-8",
)

print(
    "Local program created:",
    LOCAL_SCRIPT,
)

print(
    "Program size:",
    LOCAL_SCRIPT.stat().st_size,
    "bytes",
)

print(
    "Local syntax check:",
    "PASSED",
)


# ============================================================
# STATIC ARCHITECTURAL SAFETY CHECKS
# ============================================================

# ============================================================
# STATIC ARCHITECTURAL SAFETY CHECKS
#
# Use Python tokenization so comments and string literals do
# not create false positives.
# ============================================================

import ast
import io
import tokenize


# Confirm that the generated program parses as valid Python.
parsed_program = ast.parse(
    REMOTE_PROGRAM,
    filename=str(LOCAL_SCRIPT),
)


# ============================================================
# Extract executable Python tokens only
#
# Comments are removed before checking forbidden expressions.
# This prevents comments such as:
#
#     # no RS_BandAsArray
#
# from being mistaken for executable function calls.
# ============================================================

executable_tokens = []

token_stream = tokenize.generate_tokens(
    io.StringIO(
        REMOTE_PROGRAM
    ).readline
)

for token_info in token_stream:
    if token_info.type in {
        tokenize.COMMENT,
        tokenize.ENCODING,
    }:
        continue

    executable_tokens.append(
        token_info.string
    )

program_without_comments = tokenize.untokenize(
    [
        token_info
        for token_info in tokenize.generate_tokens(
            io.StringIO(
                REMOTE_PROGRAM
            ).readline
        )
        if token_info.type != tokenize.COMMENT
    ]
)


# ============================================================
# Check required architecture fragments
#
# These operations mostly occur inside Spark SQL strings, so
# checking the complete program text is correct here.
# ============================================================

required_fragments = [
    "RS_FromGeoTiff",
    "RS_TileExplode",
    "RS_SetBandNoDataValue",
    "RS_SummaryStats",
    'format("binaryFile")',
    ".parquet(",
]

missing_fragments = [
    fragment
    for fragment in required_fragments
    if fragment not in REMOTE_PROGRAM
]

if missing_fragments:
    raise RuntimeError(
        "Required program fragments are missing: "
        + ", ".join(missing_fragments)
    )


# ============================================================
# Check forbidden Sedona SQL function calls
#
# Match actual function syntax only:
#
#     RS_BandAsArray(
#
# A comment containing only the function name does not match.
# ============================================================

forbidden_sql_call_patterns = {
    "RS_BandAsArray call": (
        r"\bRS_BandAsArray\s*\("
    ),
    "RS_CountValue call": (
        r"\bRS_CountValue\s*\("
    ),
}

for operation, pattern in forbidden_sql_call_patterns.items():
    if re.search(
        pattern,
        program_without_comments,
    ):
        raise RuntimeError(
            f"Forbidden SQL operation found: {operation}"
        )


# ============================================================
# Inspect actual Python calls using the AST
# ============================================================

called_names = set()
called_attributes = set()

for node in ast.walk(parsed_program):
    if not isinstance(node, ast.Call):
        continue

    if isinstance(node.func, ast.Name):
        called_names.add(
            node.func.id
        )

    elif isinstance(node.func, ast.Attribute):
        called_attributes.add(
            node.func.attr
        )


forbidden_python_names = {
    "finite_statistics_udf",
}

forbidden_python_attributes = {
    "as_numpy",
    "cache",
    "persist",
    "localCheckpoint",
    "repartition",
}

found_forbidden_names = sorted(
    forbidden_python_names
    & called_names
)

found_forbidden_attributes = sorted(
    forbidden_python_attributes
    & called_attributes
)

if found_forbidden_names:
    raise RuntimeError(
        "Forbidden Python calls found: "
        + ", ".join(found_forbidden_names)
    )

if found_forbidden_attributes:
    raise RuntimeError(
        "Forbidden DataFrame/raster calls found: "
        + ", ".join(found_forbidden_attributes)
    )


# ============================================================
# Confirm that the program does not override Spark master
# ============================================================

master_override_pattern = (
    r"\.master\s*\("
)

if re.search(
    master_override_pattern,
    program_without_comments,
):
    raise RuntimeError(
        "The generated program overrides the Spark master. "
        "Remove the .master(...) call."
    )


# ============================================================
# Confirm that external raster engines are not imported
# ============================================================

imported_modules = set()

for node in ast.walk(parsed_program):
    if isinstance(node, ast.Import):
        for alias in node.names:
            imported_modules.add(
                alias.name.split(".")[0]
            )

    elif isinstance(node, ast.ImportFrom):
        if node.module:
            imported_modules.add(
                node.module.split(".")[0]
            )

forbidden_imports = {
    "numpy",
    "rasterio",
}

found_forbidden_imports = sorted(
    forbidden_imports
    & imported_modules
)

if found_forbidden_imports:
    raise RuntimeError(
        "Forbidden imports found: "
        + ", ".join(found_forbidden_imports)
    )


# ============================================================
# Final validation report
# ============================================================

print(
    "Architecture safety check:",
    "PASSED",
)

print(
    "Required architecture:",
    "HDFS -> Spark binaryFile -> Sedona raster tiles "
    "-> Spark aggregation -> HDFS Parquet",
)

print(
    "RS_BandAsArray runtime call:",
    "ABSENT",
)

print(
    "RS_CountValue runtime call:",
    "ABSENT",
)

print(
    "Python raster UDF:",
    "ABSENT",
)

print(
    "Raster-to-NumPy conversion:",
    "ABSENT",
)

print(
    "Raster shuffle/cache/checkpoint:",
    "ABSENT",
)

print(
    "Internal Spark master override:",
    "ABSENT",
)

# ============================================================
# COPY CLEAN PROGRAM TO AWS
# ============================================================

scp_result = subprocess.run(
    [
        "scp",
        "-o",
        "StrictHostKeyChecking=no",
        "-o",
        "UserKnownHostsFile=/dev/null",
        "-o",
        "BatchMode=yes",
        "-o",
        "ConnectTimeout=30",
        "-i",
        SSH_KEY,
        str(LOCAL_SCRIPT),
        f"{AWS_USER}@{AWS_HOST}:{REMOTE_SCRIPT}",
    ],
    text=True,
    capture_output=True,
)

if scp_result.stdout:
    print(scp_result.stdout)

if scp_result.stderr:
    print(scp_result.stderr)

if scp_result.returncode != 0:
    raise RuntimeError(
        "SCP failed with return code "
        f"{scp_result.returncode}."
    )

print(
    "Program copied to AWS:",
    REMOTE_SCRIPT,
)


# ============================================================
# REMOTE PRE-FLIGHT CHECK
#
# Do not call the `hdfs` CLI here.
#
# The non-interactive SSH shell does not have `hdfs` on PATH,
# but Spark accesses HDFS using:
#
#     hdfs://localhost:9000/...
#
# The generated Spark program itself verifies that exactly
# EXPECTED_SHARDS GeoTIFF files are available.
# ============================================================

preflight_command = (
    "set -e; "
    "source /home/ubuntu/ghsl-sedona-env.sh; "
    f"python3 -m py_compile {REMOTE_SCRIPT}; "
    "echo 'REMOTE PYTHON SYNTAX: PASS'; "
    "echo 'PYTHON:'; "
    "command -v python3; "
    "python3 --version; "
    "echo 'JAVA:'; "
    "java -version 2>&1 | head -n 3; "
    "echo 'SPARK-SUBMIT:'; "
    "command -v spark-submit; "
    "spark-submit --version 2>&1 | head -n 12; "
    "echo 'MEMORY:'; "
    "free -h; "
    "echo 'DISK:'; "
    "df -h / /tmp"
)

preflight_result = subprocess.run(
    [
        "bash",
        REMOTE_EXEC,
        preflight_command,
    ],
    text=True,
    capture_output=True,
    timeout=180,
)

print()
print("REMOTE PRE-FLIGHT STDOUT:")
print(preflight_result.stdout)

if preflight_result.stderr:
    print()
    print("REMOTE PRE-FLIGHT STDERR:")
    print(preflight_result.stderr)

print()
print(
    "Remote pre-flight return code:",
    preflight_result.returncode,
)

if preflight_result.returncode != 0:
    raise RuntimeError(
        "Remote pre-flight failed with return code "
        f"{preflight_result.returncode}."
    )

print()
print(
    "Remote pre-flight:",
    "PASSED",
)


# ============================================================
# CLEAN UP PREVIOUS LOG
#
# The main program writes HDFS outputs using overwrite mode,
# so no HDFS cleanup command is required.
# ============================================================

cleanup_command = (
    "rm -f "
    "/home/ubuntu/"
    "ghsl_population_1975_sedona_safe.log "
    "/home/ubuntu/"
    "ghsl_population_1975_sedona_safe.exit"
)

cleanup_result = subprocess.run(
    [
        "bash",
        REMOTE_EXEC,
        cleanup_command,
    ],
    text=True,
    capture_output=True,
    timeout=60,
)

if cleanup_result.returncode != 0:
    print(
        "Warning: remote log cleanup returned:",
        cleanup_result.returncode,
    )

    if cleanup_result.stderr:
        print(cleanup_result.stderr)


# ============================================================
# RUN CLEAN SPARK + SEDONA + HDFS PROGRAM
#
# Important:
#
# 1. The environment file is sourced before spark-submit.
# 2. The generated program contains no .master(...) call.
# 3. Therefore --master local[1] cannot be overridden.
# 4. Only one JVM raster task runs at a time.
# 5. Each source raster is split into 256 x 256 tiles before
#    RS_SummaryStats is evaluated.
# ============================================================

REMOTE_LOG = (
    "/home/ubuntu/"
    "ghsl_population_1975_sedona_safe.log"
)

REMOTE_EXIT_FILE = (
    "/home/ubuntu/"
    "ghsl_population_1975_sedona_safe.exit"
)

run_command = (
    "set -o pipefail; "
    "source /home/ubuntu/ghsl-sedona-env.sh; "
    "export PYTHONUNBUFFERED=1; "

    "spark-submit "

    "--master local[1] "

    "--driver-memory 4g "

    "--conf "
    "spark.driver.maxResultSize=512m "

    "--conf "
    "spark.sql.shuffle.partitions=1 "

    "--conf "
    "spark.default.parallelism=1 "

    "--conf "
    "spark.sql.adaptive.enabled=false "

    "--conf "
    "spark.sql.codegen.wholeStage=false "

    "--conf "
    "spark.ui.enabled=false "

    "--conf "
    "spark.network.timeout=600s "

    "--conf "
    "spark.executor.heartbeatInterval=60s "

    "--conf "
    "spark.storage.blockManagerHeartbeatTimeoutMs=600000 "

    "--conf "
    "spark.serializer="
    "org.apache.spark.serializer.KryoSerializer "

    "--packages "
    "org.apache.sedona:"
    "sedona-spark-3.5_2.12:1.9.1,"
    "org.datasyslab:"
    "geotools-wrapper:1.9.1-33.5 "

    "--repositories "
    "https://artifacts.unidata.ucar.edu/"
    "repository/unidata-all "

    f"{REMOTE_SCRIPT} "

    "2>&1 "
    f"| tee {REMOTE_LOG}; "

    "status=${PIPESTATUS[0]}; "

    f"echo $status > {REMOTE_EXIT_FILE}; "

    "echo "
    "SPARK_EXIT_CODE=${status}; "

    "exit ${status}"
)

print()
print("=" * 100)
print("STARTING CLEAN SPARK / SEDONA / HDFS JOB")
print("=" * 100)

print(
    "Remote script:",
    REMOTE_SCRIPT,
)

print(
    "Spark master:",
    "local[1]",
)

print(
    "Driver memory:",
    "4g",
)

print(
    "Raster processing:",
    "one source shard at a time",
)

print(
    "Sedona tile size:",
    "256 x 256",
)

print(
    "HDFS access:",
    "Spark Hadoop connector using hdfs://localhost:9000",
)

print(
    "HDFS CLI required:",
    "NO",
)

print(
    "Remote log:",
    REMOTE_LOG,
)


# ============================================================
# EXECUTE REMOTE JOB
# ============================================================

try:
    result = subprocess.run(
        [
            "bash",
            REMOTE_EXEC,
            run_command,
        ],
        text=True,
        capture_output=True,
        timeout=7200,
    )

except subprocess.TimeoutExpired as exc:
    print()
    print(
        "The notebook-side waiting period expired."
    )

    if exc.stdout:
        print()
        print("PARTIAL REMOTE STDOUT:")
        print(exc.stdout)

    if exc.stderr:
        print()
        print("PARTIAL REMOTE STDERR:")
        print(exc.stderr)

    timeout_diagnostics_command = (
        "echo 'RUNNING SPARK PROCESSES:'; "
        "ps -ef "
        "| grep -E "
        "'spark-submit|ghsl_population_1975_sedona_safe' "
        "| grep -v grep "
        "|| true; "

        "echo 'LAST 200 LOG LINES:'; "
        f"tail -n 200 {REMOTE_LOG} "
        "2>/dev/null "
        "|| true"
    )

    timeout_diagnostics = subprocess.run(
        [
            "bash",
            REMOTE_EXEC,
            timeout_diagnostics_command,
        ],
        text=True,
        capture_output=True,
        timeout=180,
    )

    print()
    print(timeout_diagnostics.stdout)

    if timeout_diagnostics.stderr:
        print(timeout_diagnostics.stderr)

    raise RuntimeError(
        "The remote Spark job exceeded the notebook "
        "timeout. The process and final remote log lines "
        "are printed above."
    ) from exc


# ============================================================
# DISPLAY REMOTE OUTPUT
# ============================================================

print()
print("=" * 100)
print("REMOTE JOB STDOUT")
print("=" * 100)
print(result.stdout)

if result.stderr:
    print()
    print("=" * 100)
    print("REMOTE JOB STDERR")
    print("=" * 100)
    print(result.stderr)

print()
print(
    "Remote wrapper return code:",
    result.returncode,
)


# ============================================================
# SUCCESS PATH
# ============================================================

if result.returncode == 0:
    print()
    print("=" * 100)
    print("SPARK / SEDONA / HDFS JOB COMPLETED")
    print("=" * 100)

    print(
        "Remote return code:",
        result.returncode,
    )

    print(
        "Shard statistics HDFS output:",
        "hdfs://localhost:9000"
        "/data/ghsl/processed/india/population/"
        "epoch=1975/sedona_shard_statistics",
    )

    print(
        "Epoch statistics HDFS output:",
        "hdfs://localhost:9000"
        "/data/ghsl/processed/india/population/"
        "epoch=1975/sedona_epoch_statistics",
    )

    print(
        "Full remote log:",
        REMOTE_LOG,
    )


# ============================================================
# FAILURE DIAGNOSTICS
#
# Run diagnostics only if spark-submit failed.
#
# Notice that no `hdfs` CLI command is used here.
# ============================================================

else:
    diagnostics_command = (
        "echo 'RECORDED SPARK EXIT CODE:'; "
        f"cat {REMOTE_EXIT_FILE} "
        "2>/dev/null "
        "|| echo 'exit file unavailable'; "

        "echo; "
        "echo 'LAST 250 SPARK LOG LINES:'; "
        f"tail -n 250 {REMOTE_LOG} "
        "2>/dev/null "
        "|| echo 'Spark log unavailable'; "

        "echo; "
        "echo 'CURRENT MEMORY:'; "
        "free -h "
        "|| true; "

        "echo; "
        "echo 'CURRENT DISK:'; "
        "df -h / /tmp "
        "|| true; "

        "echo; "
        "echo 'RUNNING SPARK/JAVA PROCESSES:'; "
        "ps -eo "
        "pid,ppid,%cpu,%mem,rss,vsz,etime,cmd "
        "| grep -E "
        "'spark-submit|java|"
        "ghsl_population_1975_sedona_safe' "
        "| grep -v grep "
        "| tail -n 30 "
        "|| true; "

        "echo; "
        "echo 'KERNEL OOM EVIDENCE:'; "

        "("

        "dmesg -T 2>/dev/null "
        "| grep -Ei "
        "'out of memory|oom-killer|killed process' "
        "| tail -n 40 "

        "|| "

        "journalctl -k "
        "--no-pager "
        "-n 300 "
        "2>/dev/null "
        "| grep -Ei "
        "'out of memory|oom-killer|killed process' "
        "| tail -n 40 "

        "|| true"

        ")"
    )

    diagnostics_result = subprocess.run(
        [
            "bash",
            REMOTE_EXEC,
            diagnostics_command,
        ],
        text=True,
        capture_output=True,
        timeout=240,
    )

    print()
    print("=" * 100)
    print("FAILURE DIAGNOSTICS")
    print("=" * 100)
    print(diagnostics_result.stdout)

    if diagnostics_result.stderr:
        print()
        print(
            "DIAGNOSTIC STDERR:"
        )
        print(
            diagnostics_result.stderr
        )

    raise RuntimeError(
        "The clean Spark/Sedona/HDFS job failed with "
        f"return code {result.returncode}. "
        "The final Spark log, memory state, process state "
        "and available kernel OOM evidence are printed above."
    )

In [25]:
# ============================================================
# COMPLETE CORRECTED REPLACEMENT CELL
#
# Purpose:
#   1. Load the already-generated 1975 Spark/Sedona program.
#   2. Correct RS_TileExplode aliases.
#   3. Add finite-safe raster tile statistics.
#   4. Remove NaN, +/-Inf and the -200 invalid sentinel.
#   5. Validate the corrected program locally and remotely.
#   6. Upload it to AWS.
#   7. Execute the complete 1975 job.
#
# Prerequisite:
#   /content/ghsl_population_1975_sedona_safe.py must exist.
#
# This cell replaces the previous failed replacement cell.
# ============================================================

from pathlib import Path
import re
import subprocess
import textwrap


# ============================================================
# CONFIGURATION
# ============================================================

LOCAL_SCRIPT = Path(
    "/content/ghsl_population_1975_sedona_safe.py"
)

BACKUP_SCRIPT = Path(
    "/content/ghsl_population_1975_sedona_safe_original.py"
)

REMOTE_SCRIPT = (
    "/home/ubuntu/ghsl_population_1975_sedona_safe.py"
)

REMOTE_EXEC = (
    "/content/drive/MyDrive/remote_exec.sh"
)

SSH_KEY = "/content/hadoop-hdfs.pem"

AWS_USER = "ubuntu"
AWS_HOST = "54.92.192.154"

REMOTE_LOG = (
    "/home/ubuntu/"
    "ghsl_population_1975_sedona_safe.log"
)

REMOTE_EXIT_FILE = (
    "/home/ubuntu/"
    "ghsl_population_1975_sedona_safe.exit"
)


# ============================================================
# HELPER FUNCTIONS
# ============================================================

def find_unique_line(lines, predicate, description):
    matches = [
        index
        for index, line in enumerate(lines)
        if predicate(line)
    ]

    if len(matches) != 1:
        raise RuntimeError(
            f"Expected exactly one {description}, "
            f"but found {len(matches)}."
        )

    return matches[0]


def find_next_line(
    lines,
    start_index,
    predicate,
    description,
):
    for index in range(start_index, len(lines)):
        if predicate(lines[index]):
            return index

    raise RuntimeError(
        f"Could not locate {description} after line "
        f"{start_index + 1}."
    )


def add_import_after_existing_import(
    program_text,
    import_line,
):
    """
    Add a top-level import safely.

    The generated program contains a parenthesized multiline
    import from pyspark.sql.types. Therefore, we must not insert
    a new import after an arbitrary line beginning with 'from '.
    Instead, insert immediately before the first top-level
    function or constant definition, after the complete import
    section has finished.
    """

    if import_line in program_text:
        return program_text, False

    lines = program_text.splitlines(keepends=True)

    insertion_index = None

    # Prefer inserting before the first top-level constant,
    # class, decorator or function after the import section.
    for index, line in enumerate(lines):
        if not line:
            continue

        # Ignore indented lines, including entries inside
        # parenthesized multiline imports.
        if line[0].isspace():
            continue

        stripped = line.strip()

        if (
            stripped.startswith("def ")
            or stripped.startswith("class ")
            or stripped.startswith("@")
            or re.match(
                r"^[A-Z][A-Z0-9_]*\s*=",
                stripped,
            )
        ):
            insertion_index = index
            break

    if insertion_index is None:
        raise RuntimeError(
            f"Could not find a safe top-level insertion point "
            f"for import: {import_line}"
        )

    lines.insert(
        insertion_index,
        import_line + "\n",
    )

    return "".join(lines), True


# ============================================================
# VERIFY REQUIRED FILES
# ============================================================

if not LOCAL_SCRIPT.is_file():
    raise FileNotFoundError(
        f"Generated source program not found: {LOCAL_SCRIPT}\n"
        "Run the shortened source-generation cell first."
    )

if not Path(REMOTE_EXEC).is_file():
    raise FileNotFoundError(
        f"Remote helper not found: {REMOTE_EXEC}"
    )

if not Path(SSH_KEY).is_file():
    raise FileNotFoundError(
        f"SSH key not found: {SSH_KEY}"
    )


# ============================================================
# LOAD ORIGINAL GENERATED PROGRAM
# ============================================================

program = LOCAL_SCRIPT.read_text(
    encoding="utf-8"
)

print("Loaded local program:", LOCAL_SCRIPT)
print("Original program size:", len(program), "characters")


# ============================================================
# SAVE AN UNMODIFIED BACKUP
# ============================================================

BACKUP_SCRIPT.write_text(
    program,
    encoding="utf-8",
)

print("Original source backup:", BACKUP_SCRIPT)


# ============================================================
# CORRECT RS_TileExplode ALIASES
# ============================================================

old_alias_fragments = {
    "tile_record.x AS tile_x": "tile_x",
    "tile_record.y AS tile_y": "tile_y",
    "tile_record.tile AS tile": "tile",
    "exploded AS tile_record": (
        "exploded AS tile_x, tile_y, tile"
    ),
}

old_alias_count = sum(
    old_fragment in program
    for old_fragment in old_alias_fragments
)

if old_alias_count:
    for old_fragment, new_fragment in old_alias_fragments.items():
        program = program.replace(
            old_fragment,
            new_fragment,
        )

    print("RS_TileExplode alias correction: APPLIED")

elif "exploded AS tile_x, tile_y, tile" in program:
    print("RS_TileExplode alias correction: ALREADY PRESENT")

else:
    raise RuntimeError(
        "Could not locate the old or corrected "
        "RS_TileExplode alias syntax."
    )


for old_fragment in old_alias_fragments:
    if old_fragment in program:
        raise RuntimeError(
            "Outdated RS_TileExplode fragment remains: "
            f"{old_fragment}"
        )

if "exploded AS tile_x, tile_y, tile" not in program:
    raise RuntimeError(
        "Corrected RS_TileExplode alias is absent."
    )

print("RS_TileExplode alias verification: PASS")


# ============================================================
# ADD REQUIRED IMPORTS
# ============================================================

program, numpy_added = add_import_after_existing_import(
    program,
    "import numpy as np",
)

program, udf_added = add_import_after_existing_import(
    program,
    "from pyspark.sql.functions import col, udf",
)

if numpy_added:
    print("Added import: import numpy as np")
else:
    print("NumPy import: ALREADY PRESENT")

if udf_added:
    print(
        "Added import: "
        "from pyspark.sql.functions import col, udf"
    )
else:
    print("Spark UDF imports: ALREADY PRESENT")


required_type_names = [
    "StructType",
    "StructField",
    "LongType",
    "DoubleType",
]

missing_type_names = [
    type_name
    for type_name in required_type_names
    if type_name not in program
]

if missing_type_names:
    raise RuntimeError(
        "Generated program is missing required Spark SQL types: "
        + ", ".join(missing_type_names)
    )

print("Required Spark SQL types: PRESENT")


# ============================================================
# INSERT FINITE-SAFE TILE UDF
#
# This is inserted immediately after:
#
#     shard_results = []
#
# It receives one already-created 256 x 256 Sedona raster tile.
# ============================================================

finite_udf_source = textwrap.indent(
    textwrap.dedent(
        r'''
        # ========================================================
        # FINITE-SAFE SEDONA TILE STATISTICS
        #
        # Each call receives one Sedona raster tile.
        # It excludes:
        #   - IEEE NaN
        #   - positive infinity
        #   - negative infinity
        #   - GHSL invalid sentinel -200
        # ========================================================

        finite_tile_schema = StructType(
            [
                StructField(
                    "total_count",
                    LongType(),
                    False,
                ),
                StructField(
                    "valid_count",
                    LongType(),
                    False,
                ),
                StructField(
                    "excluded_count",
                    LongType(),
                    False,
                ),
                StructField(
                    "zero_count",
                    LongType(),
                    False,
                ),
                StructField(
                    "positive_count",
                    LongType(),
                    False,
                ),
                StructField(
                    "value_sum",
                    DoubleType(),
                    False,
                ),
                StructField(
                    "value_sum_squares",
                    DoubleType(),
                    False,
                ),
                StructField(
                    "value_min",
                    DoubleType(),
                    False,
                ),
                StructField(
                    "value_max",
                    DoubleType(),
                    False,
                ),
            ]
        )

        @udf(returnType=finite_tile_schema)
        def finite_tile_statistics(raster):
            if raster is None:
                return (
                    0,
                    0,
                    0,
                    0,
                    0,
                    0.0,
                    0.0,
                    0.0,
                    0.0,
                )

            values = np.asarray(
                raster.as_numpy()[0]
            )

            total_count = int(values.size)

            valid_mask = (
                np.isfinite(values)
                & (values != INVALID_SENTINEL)
            )

            valid_values = values[valid_mask]

            valid_count = int(valid_values.size)
            excluded_count = int(
                total_count - valid_count
            )

            if valid_count == 0:
                return (
                    total_count,
                    0,
                    excluded_count,
                    0,
                    0,
                    0.0,
                    0.0,
                    0.0,
                    0.0,
                )

            valid_values = valid_values.astype(
                np.float64,
                copy=False,
            )

            negative_count = int(
                np.count_nonzero(valid_values < 0.0)
            )

            if negative_count:
                raise ValueError(
                    "Unexpected finite negative population "
                    f"values in tile: {negative_count}"
                )

            zero_count = int(
                np.count_nonzero(valid_values == 0.0)
            )

            positive_count = int(
                np.count_nonzero(valid_values > 0.0)
            )

            value_sum = float(
                np.sum(
                    valid_values,
                    dtype=np.float64,
                )
            )

            value_sum_squares = float(
                np.dot(
                    valid_values.reshape(-1),
                    valid_values.reshape(-1),
                )
            )

            value_min = float(
                np.min(valid_values)
            )

            value_max = float(
                np.max(valid_values)
            )

            calculated_values = [
                value_sum,
                value_sum_squares,
                value_min,
                value_max,
            ]

            if not all(
                math.isfinite(value)
                for value in calculated_values
            ):
                raise ValueError(
                    "Tile UDF produced a non-finite statistic."
                )

            return (
                total_count,
                valid_count,
                excluded_count,
                zero_count,
                positive_count,
                value_sum,
                value_sum_squares,
                value_min,
                value_max,
            )
        '''
    ).strip(),
    "    ",
) + "\n\n"


if "def finite_tile_statistics(raster):" in program:
    print("Finite tile UDF: ALREADY PRESENT")

else:
    lines = program.splitlines(keepends=True)

    shard_results_index = find_unique_line(
        lines,
        lambda line: line.strip() == "shard_results = []",
        "'shard_results = []' insertion line",
    )

    lines.insert(
        shard_results_index + 1,
        "\n" + finite_udf_source,
    )

    program = "".join(lines)

    print("Finite tile UDF: INSERTED")


# ============================================================
# REPLACE OLD RS_SummaryStats TILE SECTION
#
# This uses two human-readable comment labels, not an exact
# multi-line string.
# ============================================================

corrected_tile_statistics_source = textwrap.indent(
    textwrap.dedent(
        r'''
        # ----------------------------------------------------
        # Calculate finite-safe statistics for each Sedona tile
        # ----------------------------------------------------

        tile_stats_df = (
            tiles_df
            .select(
                "path",
                "source_bytes",
                "source_width",
                "source_height",
                "band_count",
                "tile_x",
                "tile_y",
                finite_tile_statistics(
                    col("tile")
                ).alias("stats"),
            )
            .select(
                "path",
                "source_bytes",
                "source_width",
                "source_height",
                "band_count",
                "tile_x",
                "tile_y",
                col("stats.total_count").alias(
                    "total_count"
                ),
                col("stats.valid_count").alias(
                    "valid_count"
                ),
                col("stats.excluded_count").alias(
                    "excluded_count"
                ),
                col("stats.zero_count").alias(
                    "zero_count"
                ),
                col("stats.positive_count").alias(
                    "positive_count"
                ),
                col("stats.value_sum").alias(
                    "value_sum"
                ),
                col("stats.value_sum_squares").alias(
                    "value_sum_squares"
                ),
                col("stats.value_min").alias(
                    "value_min"
                ),
                col("stats.value_max").alias(
                    "value_max"
                ),
            )
        )

        tile_stats_df.createOrReplaceTempView(
            "current_tile_statistics"
        )

        '''
    ).strip("\n"),
    "        ",
) + "\n"


if (
    "Calculate finite-safe statistics for each Sedona tile"
    in program
):
    print("Finite tile processing section: ALREADY PRESENT")

else:
    lines = program.splitlines(keepends=True)

    tile_section_start = find_unique_line(
        lines,
        lambda line: (
            "Calculate per-tile scalar statistics"
            in line
        ),
        "old per-tile statistics marker",
    )

    aggregate_section_start = find_next_line(
        lines,
        tile_section_start + 1,
        lambda line: (
            "Aggregate all tile statistics for this source shard"
            in line
        ),
        "shard aggregation marker",
    )

    # Include the divider line immediately before the old
    # statistics comment, when present.
    replacement_start = tile_section_start

    if (
        tile_section_start > 0
        and set(lines[tile_section_start - 1].strip())
        <= {"#", "-"}
    ):
        replacement_start = tile_section_start - 1

    # Keep the aggregation marker and everything after it.
    lines[
        replacement_start:aggregate_section_start
    ] = [corrected_tile_statistics_source]

    program = "".join(lines)

    print("Finite tile processing section: REPLACED")


# ============================================================
# ADD SHARD-LEVEL FINITE VALIDATION
#
# Insert immediately before the shard result is appended.
# This marker exists in the generated source and does not
# depend on the runtime variable's exact name.
# ============================================================

finite_shard_validation_source = textwrap.indent(
    textwrap.dedent(
        r'''
        shard_scalar_values = {
            "population_sum": population_sum,
            "sum_squares": sum_squares,
            "population_mean": population_mean,
            "population_stddev": population_stddev,
            "population_min": population_min,
            "population_max": population_max,
        }

        nonfinite_shard_values = [
            name
            for name, value in shard_scalar_values.items()
            if not math.isfinite(float(value))
        ]

        if nonfinite_shard_values:
            raise RuntimeError(
                f"Non-finite shard statistics for {filename}: "
                + ", ".join(nonfinite_shard_values)
            )

        '''
    ).strip("\n"),
    "        ",
) + "\n"


if "nonfinite_shard_values" in program:
    print("Finite shard validation: ALREADY PRESENT")

else:
    lines = program.splitlines(keepends=True)

    # Primary structural marker:
    # insert validation immediately before the current shard's
    # statistics are added to shard_results.
    append_candidates = [
        index
        for index, line in enumerate(lines)
        if (
            "shard_results.append" in line
            or "shard_results.extend" in line
        )
    ]

    if len(append_candidates) == 1:
        insertion_index = append_candidates[0]

    elif len(append_candidates) > 1:
        raise RuntimeError(
            "Found multiple shard-results append locations: "
            f"{append_candidates}. No file was modified."
        )

    else:
        # Fallback for programs that create a result Row first
        # and append it on a later line.
        result_row_candidates = [
            index
            for index, line in enumerate(lines)
            if (
                line.strip().startswith("shard_result =")
                or line.strip().startswith("result_row =")
                or line.strip().startswith("shard_row_output =")
            )
        ]

        if len(result_row_candidates) == 1:
            insertion_index = result_row_candidates[0]

        else:
            # Final fallback: place validation immediately after
            # the population_max assignment block and before the
            # next top-level statement in the shard loop.
            max_assignment_candidates = [
                index
                for index, line in enumerate(lines)
                if line.strip().startswith(
                    "population_max ="
                )
            ]

            if not max_assignment_candidates:
                raise RuntimeError(
                    "Could not locate a safe shard-validation "
                    "insertion point. No file was modified."
                )

            # Choose the last population_max assignment because
            # the source contains branches for valid and empty
            # shards.
            last_max_index = max_assignment_candidates[-1]
            insertion_index = None

            for index in range(
                last_max_index + 1,
                len(lines),
            ):
                stripped = lines[index].strip()

                if not stripped:
                    continue

                # A statement at eight-space indentation is the
                # next statement inside the per-shard loop.
                indentation = (
                    len(lines[index])
                    - len(lines[index].lstrip(" "))
                )

                if indentation == 8:
                    insertion_index = index
                    break

            if insertion_index is None:
                raise RuntimeError(
                    "Located population_max, but could not find "
                    "the following shard-loop statement."
                )

    lines.insert(
        insertion_index,
        finite_shard_validation_source,
    )

    program = "".join(lines)

    print(
        "Finite shard validation: INSERTED before line",
        insertion_index + 1,
    )


# ============================================================
# STATIC VALIDATION
# ============================================================

required_fragments = [
    "import numpy as np",
    "from pyspark.sql.functions import col, udf",
    "def finite_tile_statistics(raster):",
    "raster.as_numpy()[0]",
    "np.isfinite(values)",
    "values != INVALID_SENTINEL",
    "finite_tile_statistics(",
    'col("tile")',
    "nonfinite_shard_values",
    "exploded AS tile_x, tile_y, tile",
    "RS_FromGeoTiff(content) AS raster",
    "RS_TileExplode(",
    ".parquet(HDFS_SHARD_OUTPUT)",
    ".parquet(HDFS_EPOCH_OUTPUT)",
]

missing_fragments = [
    fragment
    for fragment in required_fragments
    if fragment not in program
]

if missing_fragments:
    raise RuntimeError(
        "Corrected program is missing required content:\n"
        + "\n".join(missing_fragments)
    )


forbidden_fragments = [
    "exploded AS tile_record",
    "tile_record.x AS tile_x",
    "tile_record.y AS tile_y",
    "tile_record.tile AS tile",
    "Calculate per-tile scalar statistics",
]

remaining_forbidden_fragments = [
    fragment
    for fragment in forbidden_fragments
    if fragment in program
]

if remaining_forbidden_fragments:
    raise RuntimeError(
        "Old content remains in corrected program:\n"
        + "\n".join(remaining_forbidden_fragments)
    )


# ============================================================
# LOCAL SYNTAX VALIDATION
# ============================================================

compile(
    program,
    str(LOCAL_SCRIPT),
    "exec",
)

print("Local corrected Python syntax: PASS")
print("Finite-value filtering: PRESENT")
print("Invalid sentinel filtering: PRESENT")
print("Finite shard validation: PRESENT")
print("RS_TileExplode alias validation: PASS")


# ============================================================
# SAVE CORRECTED SOURCE
# ============================================================

LOCAL_SCRIPT.write_text(
    program,
    encoding="utf-8",
)

print("Corrected program saved:", LOCAL_SCRIPT)
print(
    "Corrected program size:",
    LOCAL_SCRIPT.stat().st_size,
    "bytes",
)


# ============================================================
# COPY CORRECTED PROGRAM TO AWS
# ============================================================

scp_result = subprocess.run(
    [
        "scp",
        "-o",
        "StrictHostKeyChecking=no",
        "-o",
        "UserKnownHostsFile=/dev/null",
        "-o",
        "BatchMode=yes",
        "-o",
        "ConnectTimeout=30",
        "-i",
        SSH_KEY,
        str(LOCAL_SCRIPT),
        f"{AWS_USER}@{AWS_HOST}:{REMOTE_SCRIPT}",
    ],
    text=True,
    capture_output=True,
)

if scp_result.stdout:
    print(scp_result.stdout)

if scp_result.stderr:
    print(scp_result.stderr)

if scp_result.returncode != 0:
    raise RuntimeError(
        "SCP failed with return code "
        f"{scp_result.returncode}."
    )

print("Corrected program copied to AWS:", REMOTE_SCRIPT)


# ============================================================
# REMOTE PRE-FLIGHT
# ============================================================

preflight_command = (
    "set -e; "
    "source /home/ubuntu/ghsl-sedona-env.sh; "
    f"python3 -m py_compile {REMOTE_SCRIPT}; "
    "echo 'REMOTE PYTHON SYNTAX: PASS'; "
    "echo 'PYTHON:'; "
    "python3 --version; "
    "echo 'NUMPY:'; "
    "python3 -c 'import numpy; print(numpy.__version__)'; "
    "echo 'PYSPARK:'; "
    "python3 -c 'import pyspark; print(pyspark.__version__)'; "
    "echo 'SEDONA:'; "
    "python3 -c "
    "'from importlib.metadata import version; "
    "print(version(\"apache-sedona\"))'; "
    "echo 'MEMORY:'; "
    "free -h; "
    "echo 'DISK:'; "
    "df -h / /tmp"
)

preflight_result = subprocess.run(
    [
        "bash",
        REMOTE_EXEC,
        preflight_command,
    ],
    text=True,
    capture_output=True,
    timeout=240,
)

print()
print("=" * 100)
print("REMOTE PRE-FLIGHT")
print("=" * 100)
print(preflight_result.stdout)

if preflight_result.stderr:
    print("REMOTE PRE-FLIGHT STDERR:")
    print(preflight_result.stderr)

print(
    "Remote pre-flight return code:",
    preflight_result.returncode,
)

if preflight_result.returncode != 0:
    raise RuntimeError(
        "Remote pre-flight failed with return code "
        f"{preflight_result.returncode}."
    )


# ============================================================
# CLEAN OLD LOG FILES
# ============================================================

cleanup_result = subprocess.run(
    [
        "bash",
        REMOTE_EXEC,
        f"rm -f {REMOTE_LOG} {REMOTE_EXIT_FILE}",
    ],
    text=True,
    capture_output=True,
    timeout=60,
)

if cleanup_result.returncode != 0:
    print("Warning: old log cleanup failed.")
    print(cleanup_result.stderr)


# ============================================================
# BUILD SPARK COMMAND
# ============================================================

run_command = (
    "set -o pipefail; "
    "source /home/ubuntu/ghsl-sedona-env.sh; "
    "export PYTHONUNBUFFERED=1; "

    "spark-submit "
    "--master local[1] "
    "--driver-memory 4g "

    "--conf spark.driver.maxResultSize=512m "
    "--conf spark.sql.shuffle.partitions=1 "
    "--conf spark.default.parallelism=1 "
    "--conf spark.sql.adaptive.enabled=false "
    "--conf spark.sql.codegen.wholeStage=false "
    "--conf spark.ui.enabled=false "
    "--conf spark.network.timeout=600s "
    "--conf spark.executor.heartbeatInterval=60s "
    "--conf spark.storage.blockManagerHeartbeatTimeoutMs=600000 "
    "--conf spark.python.worker.reuse=true "

    "--conf "
    "spark.serializer="
    "org.apache.spark.serializer.KryoSerializer "

    "--packages "
    "org.apache.sedona:"
    "sedona-spark-3.5_2.12:1.9.1,"
    "org.datasyslab:"
    "geotools-wrapper:1.9.1-33.5 "

    "--repositories "
    "https://artifacts.unidata.ucar.edu/"
    "repository/unidata-all "

    f"{REMOTE_SCRIPT} "

    "2>&1 "
    f"| tee {REMOTE_LOG}; "

    "status=${PIPESTATUS[0]}; "
    f"echo $status > {REMOTE_EXIT_FILE}; "
    "echo SPARK_EXIT_CODE=${status}; "
    "exit ${status}"
)


# ============================================================
# RUN CORRECTED JOB
# ============================================================

print()
print("=" * 100)
print("STARTING CORRECTED 1975 SPARK / SEDONA / HDFS JOB")
print("=" * 100)
print("AWS host:", AWS_HOST)
print("Spark master: local[1]")
print("Driver memory: 4g")
print("Validity rule: finite value AND value != -200")
print("Sedona tile size: 256 x 256")
print("Remote script:", REMOTE_SCRIPT)
print("Remote log:", REMOTE_LOG)
print()
print(
    "Existing invalid 1975 HDFS statistics will be "
    "overwritten by the corrected program."
)


try:
    result = subprocess.run(
        [
            "bash",
            REMOTE_EXEC,
            run_command,
        ],
        text=True,
        capture_output=True,
        timeout=14400,
    )

except subprocess.TimeoutExpired as exc:
    print()
    print(
        "The Colab waiting period expired after four hours."
    )

    if exc.stdout:
        print("PARTIAL STDOUT:")
        print(exc.stdout)

    if exc.stderr:
        print("PARTIAL STDERR:")
        print(exc.stderr)

    status_command = (
        "echo 'RUNNING PROCESSES:'; "
        "ps -ef "
        "| grep -E "
        "'spark-submit|"
        "ghsl_population_1975_sedona_safe' "
        "| grep -v grep "
        "|| true; "
        "echo; "
        "echo 'LAST 250 LOG LINES:'; "
        f"tail -n 250 {REMOTE_LOG} "
        "2>/dev/null "
        "|| true"
    )

    status_result = subprocess.run(
        [
            "bash",
            REMOTE_EXEC,
            status_command,
        ],
        text=True,
        capture_output=True,
        timeout=180,
    )

    print(status_result.stdout)

    if status_result.stderr:
        print(status_result.stderr)

    raise RuntimeError(
        "The Colab wait timed out. The remote Spark job may "
        "still be running. Check the process and log output above."
    ) from exc


# ============================================================
# DISPLAY RESULT
# ============================================================

print()
print("=" * 100)
print("REMOTE JOB STDOUT")
print("=" * 100)
print(result.stdout)

if result.stderr:
    print()
    print("=" * 100)
    print("REMOTE JOB STDERR")
    print("=" * 100)
    print(result.stderr)

print()
print("Remote return code:", result.returncode)


# ============================================================
# SUCCESS OR FAILURE HANDLING
# ============================================================

if result.returncode == 0:
    print()
    print("=" * 100)
    print("CORRECTED 1975 JOB COMPLETED SUCCESSFULLY")
    print("=" * 100)

    print(
        "Shard output:",
        "hdfs://localhost:9000"
        "/data/ghsl/processed/india/population/"
        "epoch=1975/sedona_shard_statistics",
    )

    print(
        "Epoch output:",
        "hdfs://localhost:9000"
        "/data/ghsl/processed/india/population/"
        "epoch=1975/sedona_epoch_statistics",
    )

    print("Remote log:", REMOTE_LOG)

else:
    diagnostics_command = (
        "echo 'RECORDED EXIT CODE:'; "
        f"cat {REMOTE_EXIT_FILE} "
        "2>/dev/null "
        "|| echo 'exit file unavailable'; "

        "echo; "
        "echo 'ERROR LOCATIONS:'; "
        "grep -n -E "
        "'Traceback|"
        "AnalysisException|"
        "ParseException|"
        "PythonException|"
        "Py4JJavaError|"
        "OutOfMemoryError|"
        "ValueError|"
        "RuntimeError|"
        "Caused by:|"
        "JOB FAILED' "
        f"{REMOTE_LOG} "
        "2>/dev/null "
        "| head -n 150 "
        "|| true; "

        "echo; "
        "echo 'LAST 350 LOG LINES:'; "
        f"tail -n 350 {REMOTE_LOG} "
        "2>/dev/null "
        "|| true; "

        "echo; "
        "echo 'MEMORY:'; "
        "free -h "
        "|| true; "

        "echo; "
        "echo 'DISK:'; "
        "df -h / /tmp "
        "|| true"
    )

    diagnostics_result = subprocess.run(
        [
            "bash",
            REMOTE_EXEC,
            diagnostics_command,
        ],
        text=True,
        capture_output=True,
        timeout=300,
    )

    print()
    print("=" * 100)
    print("FAILURE DIAGNOSTICS")
    print("=" * 100)
    print(diagnostics_result.stdout)

    if diagnostics_result.stderr:
        print("DIAGNOSTIC STDERR:")
        print(diagnostics_result.stderr)

    raise RuntimeError(
        "Corrected Spark/Sedona job failed with return code "
        f"{result.returncode}. See diagnostics above."
    )

Loaded local program: /content/ghsl_population_1975_sedona_safe.py
Original program size: 27979 characters
Original source backup: /content/ghsl_population_1975_sedona_safe_original.py
RS_TileExplode alias correction: ALREADY PRESENT
RS_TileExplode alias verification: PASS
NumPy import: ALREADY PRESENT
Spark UDF imports: ALREADY PRESENT
Required Spark SQL types: PRESENT
Finite tile UDF: ALREADY PRESENT
Finite tile processing section: ALREADY PRESENT
Finite shard validation: ALREADY PRESENT
Local corrected Python syntax: PASS
Finite-value filtering: PRESENT
Invalid sentinel filtering: PRESENT
Finite shard validation: PRESENT
RS_TileExplode alias validation: PASS
Corrected program saved: /content/ghsl_population_1975_sedona_safe.py
Corrected program size: 27979 bytes

Corrected program copied to AWS: /home/ubuntu/ghsl_population_1975_sedona_safe.py

REMOTE PRE-FLIGHT
REMOTE PYTHON SYNTAX: PASS
PYTHON:
Python 3.12.16
NUMPY:
2.5.4
PYSPARK:
3.5.9
SEDONA:
1.9.1
MEMORY:
               total  

RuntimeError: Corrected Spark/Sedona job failed with return code 137. See diagnostics above.

In [26]:
# ============================================================
# MEMORY-SAFE RERUN OF THE ALREADY-CORRECTED 1975 PROGRAM
#
# Do not rerun the large patching cell.
# This uses the corrected program already uploaded to AWS.
# ============================================================

import subprocess

REMOTE_EXEC = "/content/drive/MyDrive/remote_exec.sh"

REMOTE_SCRIPT = (
    "/home/ubuntu/ghsl_population_1975_sedona_safe.py"
)

REMOTE_LOG = (
    "/home/ubuntu/"
    "ghsl_population_1975_sedona_memory_safe.log"
)

REMOTE_EXIT_FILE = (
    "/home/ubuntu/"
    "ghsl_population_1975_sedona_memory_safe.exit"
)

SPARK_LOCAL_DIR = (
    "/home/ubuntu/spark-local-ghsl"
)


# ============================================================
# 1. VERIFY THAT THE FAILED JOB IS NOT STILL RUNNING
# ============================================================

process_check_command = (
    "ps -ef "
    "| grep -E "
    "'spark-submit|"
    "ghsl_population_1975_sedona_safe.py' "
    "| grep -v grep "
    "|| true"
)

process_check = subprocess.run(
    [
        "bash",
        REMOTE_EXEC,
        process_check_command,
    ],
    text=True,
    capture_output=True,
    timeout=60,
)

running_processes = process_check.stdout.strip()

if running_processes:
    print("Unexpected existing Spark process:")
    print(running_processes)

    raise RuntimeError(
        "An older Spark job is still running. "
        "Do not start another copy."
    )

print("Existing Spark process: NONE")


# ============================================================
# 2. CREATE OR ENABLE 8 GiB SWAP
#
# This is stored on the EC2 root EBS volume.
# The output showed approximately 61 GiB free disk space.
# ============================================================

swap_setup_command = (
    "set -e; "

    "if swapon --show=NAME --noheadings "
    "| grep -qx '/swapfile'; "
    "then "
    "echo 'Swap file already enabled'; "

    "else "
    "echo 'Preparing 8 GiB swap file'; "

    "sudo swapoff /swapfile "
    "2>/dev/null "
    "|| true; "

    "if [ ! -f /swapfile ]; "
    "then "
    "sudo fallocate -l 8G /swapfile "
    "|| sudo dd "
    "if=/dev/zero "
    "of=/swapfile "
    "bs=1M "
    "count=8192 "
    "status=progress; "
    "fi; "

    "sudo chmod 600 /swapfile; "
    "sudo mkswap /swapfile; "
    "sudo swapon /swapfile; "
    "fi; "

    "echo; "
    "echo 'ACTIVE SWAP:'; "
    "swapon --show; "

    "echo; "
    "echo 'MEMORY AFTER SWAP SETUP:'; "
    "free -h; "

    "echo; "
    "echo 'DISK AFTER SWAP SETUP:'; "
    "df -h /"
)

swap_setup = subprocess.run(
    [
        "bash",
        REMOTE_EXEC,
        swap_setup_command,
    ],
    text=True,
    capture_output=True,
    timeout=900,
)

print(swap_setup.stdout)

if swap_setup.stderr:
    print("SWAP SETUP STDERR:")
    print(swap_setup.stderr)

if swap_setup.returncode != 0:
    raise RuntimeError(
        "Swap setup failed with return code "
        f"{swap_setup.returncode}."
    )

print("Swap setup: PASS")


# ============================================================
# 3. CREATE DISK-BACKED SPARK TEMPORARY DIRECTORY
#
# Avoid /tmp because /tmp is memory-backed on this EC2 host.
# ============================================================

temp_setup_command = (
    "set -e; "
    f"mkdir -p {SPARK_LOCAL_DIR}; "
    f"rm -rf {SPARK_LOCAL_DIR}/*; "
    f"chmod 700 {SPARK_LOCAL_DIR}; "
    f"ls -ld {SPARK_LOCAL_DIR}; "
    "df -h / /tmp"
)

temp_setup = subprocess.run(
    [
        "bash",
        REMOTE_EXEC,
        temp_setup_command,
    ],
    text=True,
    capture_output=True,
    timeout=120,
)

print(temp_setup.stdout)

if temp_setup.stderr:
    print("TEMP SETUP STDERR:")
    print(temp_setup.stderr)

if temp_setup.returncode != 0:
    raise RuntimeError(
        "Spark temporary-directory setup failed."
    )

print("Disk-backed Spark temporary directory: READY")


# ============================================================
# 4. REMOVE ONLY THE NEW RERUN LOG FILES
# ============================================================

cleanup_command = (
    f"rm -f {REMOTE_LOG} {REMOTE_EXIT_FILE}"
)

cleanup_result = subprocess.run(
    [
        "bash",
        REMOTE_EXEC,
        cleanup_command,
    ],
    text=True,
    capture_output=True,
    timeout=60,
)

if cleanup_result.returncode != 0:
    print("Warning: log cleanup was unsuccessful.")
    print(cleanup_result.stderr)


# ============================================================
# 5. BUILD MEMORY-SAFE SPARK COMMAND
#
# Changes from failed run:
#   - driver memory: 4g -> 3g
#   - driver max result: 512m -> 256m
#   - Spark local storage moved off /tmp tmpfs
#   - Python worker memory limited to 1g
#   - 8 GiB swap available as emergency headroom
# ============================================================

run_command = (
    "set -o pipefail; "

    "source /home/ubuntu/ghsl-sedona-env.sh; "

    "export PYTHONUNBUFFERED=1; "
    f"export SPARK_LOCAL_DIRS={SPARK_LOCAL_DIR}; "

    "spark-submit "

    "--master local[1] "

    "--driver-memory 3g "

    "--conf spark.driver.maxResultSize=256m "
    "--conf spark.local.dir="
    f"{SPARK_LOCAL_DIR} "

    "--conf spark.sql.shuffle.partitions=1 "
    "--conf spark.default.parallelism=1 "
    "--conf spark.sql.adaptive.enabled=false "
    "--conf spark.sql.codegen.wholeStage=false "
    "--conf spark.ui.enabled=false "

    "--conf spark.network.timeout=600s "
    "--conf spark.executor.heartbeatInterval=60s "
    "--conf "
    "spark.storage.blockManagerHeartbeatTimeoutMs=600000 "

    "--conf spark.python.worker.reuse=true "
    "--conf spark.python.worker.memory=1g "

    "--conf spark.memory.fraction=0.45 "
    "--conf spark.memory.storageFraction=0.20 "

    "--conf "
    "spark.serializer="
    "org.apache.spark.serializer.KryoSerializer "

    "--packages "
    "org.apache.sedona:"
    "sedona-spark-3.5_2.12:1.9.1,"
    "org.datasyslab:"
    "geotools-wrapper:1.9.1-33.5 "

    "--repositories "
    "https://artifacts.unidata.ucar.edu/"
    "repository/unidata-all "

    f"{REMOTE_SCRIPT} "

    "2>&1 "
    f"| tee {REMOTE_LOG}; "

    "status=${PIPESTATUS[0]}; "
    f"echo $status > {REMOTE_EXIT_FILE}; "

    "echo SPARK_EXIT_CODE=${status}; "
    "exit ${status}"
)


# ============================================================
# 6. EXECUTE THE ALREADY-CORRECTED PROGRAM
# ============================================================

print()
print("=" * 100)
print("STARTING MEMORY-SAFE CORRECTED 1975 JOB")
print("=" * 100)
print("Corrected remote program:", REMOTE_SCRIPT)
print("Driver memory: 3g")
print("Python worker memory: 1g")
print("Spark local directory:", SPARK_LOCAL_DIR)
print("Remote log:", REMOTE_LOG)
print("Expected shards: 15")
print()


try:
    result = subprocess.run(
        [
            "bash",
            REMOTE_EXEC,
            run_command,
        ],
        text=True,
        capture_output=True,
        timeout=21600,
    )

except subprocess.TimeoutExpired as exc:
    print(
        "Colab stopped waiting after six hours. "
        "The remote process may still be running."
    )

    if exc.stdout:
        print("PARTIAL STDOUT:")
        print(exc.stdout)

    if exc.stderr:
        print("PARTIAL STDERR:")
        print(exc.stderr)

    status_command = (
        "echo 'RUNNING PROCESSES:'; "
        "ps -ef "
        "| grep -E "
        "'spark-submit|"
        "ghsl_population_1975_sedona_safe.py' "
        "| grep -v grep "
        "|| true; "

        "echo; "
        "echo 'LATEST LOG:'; "
        f"tail -n 250 {REMOTE_LOG} "
        "2>/dev/null "
        "|| true; "

        "echo; "
        "echo 'MEMORY:'; "
        "free -h; "

        "echo; "
        "echo 'SWAP:'; "
        "swapon --show"
    )

    status_result = subprocess.run(
        [
            "bash",
            REMOTE_EXEC,
            status_command,
        ],
        text=True,
        capture_output=True,
        timeout=180,
    )

    print(status_result.stdout)

    if status_result.stderr:
        print(status_result.stderr)

    raise RuntimeError(
        "Notebook wait timed out. Inspect the process and "
        "remote log output printed above."
    ) from exc


# ============================================================
# 7. DISPLAY RESULT
# ============================================================

print()
print("=" * 100)
print("REMOTE JOB OUTPUT")
print("=" * 100)
print(result.stdout)

if result.stderr:
    print()
    print("REMOTE STDERR:")
    print(result.stderr)

print()
print("Remote return code:", result.returncode)


# ============================================================
# 8. SUCCESS OR FAILURE DIAGNOSTICS
# ============================================================

if result.returncode == 0:
    print()
    print("=" * 100)
    print("MEMORY-SAFE CORRECTED 1975 JOB: SUCCESS")
    print("=" * 100)
    print("All corrected HDFS statistics were overwritten.")
    print("Remote log:", REMOTE_LOG)

else:
    diagnostics_command = (
        "echo 'EXIT CODE:'; "
        f"cat {REMOTE_EXIT_FILE} "
        "2>/dev/null "
        "|| true; "

        "echo; "
        "echo 'LAST 300 LOG LINES:'; "
        f"tail -n 300 {REMOTE_LOG} "
        "2>/dev/null "
        "|| true; "

        "echo; "
        "echo 'MEMORY:'; "
        "free -h; "

        "echo; "
        "echo 'SWAP:'; "
        "swapon --show; "

        "echo; "
        "echo 'DISK:'; "
        "df -h / /tmp; "

        "echo; "
        "echo 'KERNEL OOM MESSAGES:'; "
        "sudo dmesg -T "
        "| grep -Ei "
        "'out of memory|oom-killer|killed process' "
        "| tail -n 50 "
        "|| true"
    )

    diagnostics = subprocess.run(
        [
            "bash",
            REMOTE_EXEC,
            diagnostics_command,
        ],
        text=True,
        capture_output=True,
        timeout=300,
    )

    print()
    print("=" * 100)
    print("FAILURE DIAGNOSTICS")
    print("=" * 100)
    print(diagnostics.stdout)

    if diagnostics.stderr:
        print("DIAGNOSTIC STDERR:")
        print(diagnostics.stderr)

    raise RuntimeError(
        "The memory-safe corrected job failed with return code "
        f"{result.returncode}. See diagnostics above."
    )

Existing Spark process: NONE
Preparing 8 GiB swap file
Setting up swapspace version 1, size = 8 GiB (8589930496 bytes)
no label, UUID=4bae5ef2-b63f-4d2d-81b9-81e679d980df

ACTIVE SWAP:
NAME      TYPE SIZE USED PRIO
/swapfile file   8G   0B   -1

MEMORY AFTER SWAP SETUP:
               total        used        free      shared  buff/cache   available
Mem:           7.6Gi       2.9Gi       4.7Gi       1.6Gi       1.8Gi       4.7Gi
Swap:          8.0Gi          0B       8.0Gi

DISK AFTER SWAP SETUP:
Filesystem      Size  Used Avail Use% Mounted on
/dev/root        77G   24G   54G  31% /

SWAP SETUP STDERR:

Swap setup: PASS
drwx------ 2 ubuntu ubuntu 4096 Nov  3 15:45 /home/ubuntu/spark-local-ghsl
Filesystem      Size  Used Avail Use% Mounted on
/dev/root        77G   24G   54G  31% /
tmpfs           3.8G  1.7G  2.2G  43% /tmp

TEMP SETUP STDERR:

Disk-backed Spark temporary directory: READY

STARTING MEMORY-SAFE CORRECTED 1975 JOB
Corrected remote program: /home/ubuntu/ghsl_population_19

RuntimeError: Notebook wait timed out. Inspect the process and remote log output printed above.